# Drug-Target Interaction Prediction Model
This notebook contains the complete pipeline for DTI prediction using PyTorch Geometric.

**Install (pick one):**
- **Cell 1** — Kaggle / pip-only environments
- **Cells 2–3** — Local **Conda + NVIDIA GPU** (e.g. RTX 4060 Ti). Skip cell 1 when using this path.

In [ ]:
# [KAGGLE ONLY] Install dependencies — auto-skipped on local machine
import importlib
import os
import subprocess
import sys

ON_KAGGLE = os.path.exists("/kaggle/input") or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if not ON_KAGGLE:
    print("Not on Kaggle — skipping pip installs. Run LOCAL setup cells instead.")
    print("Python:", sys.executable)
else:
    def ensure_pip(pkg, import_name=None):
        name = import_name or pkg.replace("-", "_")
        try:
            importlib.import_module(name)
            print(f"OK: {name}")
        except ImportError:
            print(f"Installing {pkg} ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            importlib.import_module(name)
            print(f"Installed {pkg}")

    for pip_pkg, mod in [
        ("prefetch_generator", "prefetch_generator"),
        ("pyarrow", "pyarrow"),
        ("fair-esm", "esm"),
        ("rdkit", "rdkit"),
    ]:
        ensure_pip(pip_pkg, mod)

    try:
        importlib.import_module("torch_geometric")
        print("OK: torch_geometric")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
        print("Installed torch-geometric")

    from rdkit import Chem
    assert Chem.MolFromSmiles("CCO") is not None
    print("RDKit sanity check passed")


### Local Conda setup (RTX 4060 Ti / other NVIDIA GPUs)

**Do not run `conda install` inside the notebook** — on Windows it often **kills the Jupyter kernel**. Install everything once in **Anaconda Prompt**, then run only the **check** code cell below.

**One-time** (Anaconda Prompt):

```bash
conda create -n dti_research python=3.11 -y
conda activate dti_research
conda install pytorch pytorch-cuda=12.4 -c pytorch -c nvidia -y
conda install -c conda-forge rdkit pyarrow scikit-learn -y
pip install fair-esm torch-geometric prefetch_generator pandas tqdm joblib pyarrow
python -m ipykernel install --user --name dti_research
```

Optional (faster PyG scatter; match your torch/CUDA — run in Python if unsure):

```python
import torch
tv = ".".join(torch.__version__.split(".")[:2])
cu = "cu" + torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu"
print(f"pip install torch-scatter -f https://data.pyg.org/whl/torch-{tv}+{cu}.html")
```

Then: select kernel **`dti_research`**, open this notebook and run in order:

1. **LOCAL Step A** (paths only)  
2. **LOCAL Step B** (subprocess diagnostics — reports which package breaks)  
3. **Imports** cell and the rest  

Or run `setup_local.bat` from this repo folder in Anaconda Prompt (creates env from `environment.yml`).

**Skip** the Kaggle-only cell (auto-skips locally anyway).

In [24]:
# [LOCAL] Step A — paths only (run this first; should never crash)
import os
import sys

os.environ["MIF_DTI_ENV"] = "local_conda"
LOCAL_DATA_DIR = os.path.join(os.getcwd(), "data")
LOCAL_SAVE_DIR = os.path.join(os.getcwd(), "dti_run")
os.makedirs(LOCAL_SAVE_DIR, exist_ok=True)

# Windows: avoid OpenMP crash when importing torch (multiple MKL/OpenMP copies)
if os.name == "nt":
    os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
    os.environ["OMP_NUM_THREADS"] = "1"

print("Local mode enabled.")
print("Python:", sys.executable)
print("DATA ->", LOCAL_DATA_DIR)
print("SAVE ->", LOCAL_SAVE_DIR)
print("Next: run the diagnostic cell below (Step B).")


Local mode enabled.
Python: c:\Users\Abdullah\anaconda3\envs\dti_research\python.exe
DATA -> d:\Thesis\DTI_Models\Runners\MIF-DTI\data
SAVE -> d:\Thesis\DTI_Models\Runners\MIF-DTI\dti_run
Next: run the diagnostic cell below (Step B).


In [25]:
# [LOCAL] Step B — test dti_research env (subprocess; kernel stays alive on failure)
import os
import shutil
import subprocess
import sys

CONDA_ENV = "dti_research"


def resolve_dti_python():
    """Use conda env python even if Jupyter picked a different kernel."""
    kernel_py = sys.executable
    conda_exe = shutil.which("conda")
    env_py = None

    if conda_exe:
        r = subprocess.run(
            [conda_exe, "run", "-n", CONDA_ENV, "python", "-c", "import sys; print(sys.executable)"],
            capture_output=True,
            text=True,
            timeout=120,
        )
        if r.returncode == 0:
            env_py = r.stdout.strip().splitlines()[-1]

    if env_py is None:
        print("Could not resolve conda env via 'conda run'. Using kernel Python:")
        print(" ", kernel_py)
        return kernel_py, False

    same = os.path.normcase(env_py) == os.path.normcase(kernel_py)
    print("dti_research Python:", env_py)
    print("Jupyter kernel Python:", kernel_py)

    if not same:
        print("\n*** KERNEL MISMATCH ***")
        print("Packages you installed in dti_research are NOT visible to this kernel.")
        print("Fix: Jupyter/VS Code -> select kernel 'Python (dti_research)' -> Restart kernel")
        print("Then re-run Step A and Step B.\n")
    else:
        print("Kernel matches dti_research.")

    return env_py, same


PYTHON, KERNEL_MATCHES = resolve_dti_python()

# Windows OpenMP: torch/esm fail without this when MKL + PyTorch both linked
SUBPROCESS_ENV = os.environ.copy()
if os.name == "nt":
    SUBPROCESS_ENV["KMP_DUPLICATE_LIB_OK"] = "TRUE"
    SUBPROCESS_ENV["OMP_NUM_THREADS"] = "1"
    _IMPORT_PREFIX = (
        "import os; os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'; "
        "os.environ['OMP_NUM_THREADS']='1'; "
    )
else:
    _IMPORT_PREFIX = ""


def test_import(mod):
    code = _IMPORT_PREFIX + f"import {mod}; print('ok')"
    return subprocess.run(
        [PYTHON, "-c", code],
        capture_output=True,
        text=True,
        timeout=180,
        env=SUBPROCESS_ENV,
    )


MODULES = {
    "numpy": "conda activate dti_research && conda install numpy",
    "pandas": "pip install pandas",
    "sklearn": "conda install -c conda-forge scikit-learn",
    "torch": "conda install pytorch pytorch-cuda=12.4 -c pytorch -c nvidia",
    "torch_geometric": "pip install torch-geometric",
    "pyarrow": "conda install -c conda-forge pyarrow",
    "joblib": "pip install joblib",
    "tqdm": "pip install tqdm",
    "prefetch_generator": "pip install prefetch_generator",
    "rdkit": "conda install -c conda-forge rdkit",
    "esm": "pip install fair-esm",
}

failed = []
for mod, fix in MODULES.items():
    p = test_import(mod)
    if p.returncode == 0:
        print(f"OK   {mod}")
    else:
        print(f"FAIL {mod}")
        err = (p.stderr or p.stdout or "").strip().splitlines()
        if err:
            print("     ", err[-1][:300])
        print(f"     fix (in Anaconda Prompt): conda activate {CONDA_ENV} && {fix}")
        failed.append(mod)

if failed:
    if os.name == "nt" and set(failed) <= {"torch", "torch_geometric", "esm"}:
        print("\nOpenMP / DLL issue on Windows. Run fix_dti_research_env.bat in Anaconda Prompt,")
        print("restart kernel, then re-run Step A and Step B.\n")
    raise RuntimeError(
        f"{len(failed)} module(s) failed in {CONDA_ENV}: {failed}\n"
        "See fix_dti_research_env.bat or markdown troubleshooting above."
    )

if not KERNEL_MATCHES:
    raise RuntimeError(
        "dti_research packages look OK, but Jupyter is using a different Python.\n"
        "Select kernel 'Python (dti_research)', restart kernel, re-run Step A & B before training."
    )

gpu_cmd = (
    _IMPORT_PREFIX
    + "import torch; "
    "assert torch.cuda.is_available(), 'CUDA not available'; "
    "p=torch.cuda.get_device_properties(0); "
    "print(torch.__version__); print(torch.version.cuda); "
    "print(torch.cuda.get_device_name(0)); print(p.total_memory/1024**3)"
)
p = subprocess.run(
    [PYTHON, "-c", gpu_cmd],
    capture_output=True,
    text=True,
    timeout=120,
    env=SUBPROCESS_ENV,
)
if p.returncode != 0:
    print(p.stderr or p.stdout)
    raise RuntimeError("CUDA check failed in dti_research. Reinstall GPU PyTorch in that env.")

lines = p.stdout.strip().splitlines()
print(f"\nPyTorch {lines[0]} | CUDA {lines[1]}")
print(f"GPU: {lines[2]} | VRAM: {float(lines[3]):.1f} GB")
vram_gb = float(lines[3])
SUGGESTED_BATCH_SIZE = 8 if vram_gb >= 15 else 4
SUGGESTED_GRAD_ACCUM = 16 if vram_gb >= 15 else 32
print(f"Suggested: BATCH_SIZE={SUGGESTED_BATCH_SIZE}, GRAD_ACCUM_STEPS={SUGGESTED_GRAD_ACCUM}")
print("Diagnostics passed — continue with imports cell.")


dti_research Python: C:\Users\Abdullah\anaconda3\envs\dti_research\python.exe
Jupyter kernel Python: c:\Users\Abdullah\anaconda3\envs\dti_research\python.exe
Kernel matches dti_research.
OK   numpy
OK   pandas
OK   sklearn
OK   torch
OK   torch_geometric
OK   pyarrow
OK   joblib
OK   tqdm
OK   prefetch_generator
OK   rdkit
OK   esm

PyTorch 2.5.1 | CUDA 12.4
GPU: NVIDIA GeForce RTX 4060 Ti | VRAM: 8.0 GB
Suggested: BATCH_SIZE=4, GRAD_ACCUM_STEPS=32
Diagnostics passed — continue with imports cell.


In [77]:
# Import libraries
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data

from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, average_precision_score, auc, precision_recall_curve,
    precision_score, recall_score, roc_auc_score
)
try:
    from prefetch_generator import BackgroundGenerator
except ImportError:
    # Fallback when prefetch_generator is unavailable (e.g. failed pip on Kaggle)
    class BackgroundGenerator:
        def __init__(self, generator):
            self.generator = generator
        def __iter__(self):
            return iter(self.generator)

from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4060 Ti


In [78]:
import torch
from torch.utils.data import Dataset
import numpy as np
CHARISOSMISET = {"#": 29, "%": 30, ")": 31, "(": 1, "+": 32, "-": 33, "/": 34, ".": 2,
                 "1": 35, "0": 3, "3": 36, "2": 4, "5": 37, "4": 5, "7": 38, "6": 6,
                 "9": 39, "8": 7, "=": 40, "A": 41, "@": 8, "C": 42, "B": 9, "E": 43,
                 "D": 10, "G": 44, "F": 11, "I": 45, "H": 12, "K": 46, "M": 47, "L": 13,
                 "O": 48, "N": 14, "P": 15, "S": 49, "R": 16, "U": 50, "T": 17, "W": 51,
                 "V": 18, "Y": 52, "[": 53, "Z": 19, "]": 54, "\\": 20, "a": 55, "c": 56,
                 "b": 21, "e": 57, "d": 22, "g": 58, "f": 23, "i": 59, "h": 24, "m": 60,
                 "l": 25, "o": 61, "n": 26, "s": 62, "r": 27, "u": 63, "t": 28, "y": 64}

CHARISOSMILEN = 64

CHARPROTSET = {"A": 1, "C": 2, "B": 3, "E": 4, "D": 5, "G": 6,
               "F": 7, "I": 8, "H": 9, "K": 10, "M": 11, "L": 12,
               "O": 13, "N": 14, "Q": 15, "P": 16, "S": 17, "R": 18,
               "U": 19, "T": 20, "W": 21, "V": 22, "Y": 23, "X": 24, "Z": 25}

CHARPROTLEN = 25


def label_smiles(line, smi_ch_ind=CHARISOSMISET, MAX_SMI_LEN=100):
    X = np.zeros(MAX_SMI_LEN, dtype=np.int64())
    for i, ch in enumerate(line[:MAX_SMI_LEN]):
        X[i] = smi_ch_ind[ch]
    return X


def label_sequence(line, smi_ch_ind=CHARPROTSET, MAX_SEQ_LEN=1000):
    X = np.zeros(MAX_SEQ_LEN, np.int64())
    for i, ch in enumerate(line[:MAX_SEQ_LEN]):
        X[i] = smi_ch_ind[ch]
    return X


class CustomDataSet(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __getitem__(self, item):
        return self.pairs[item]

    def __len__(self):
        return len(self.pairs)


def collate_fn(batch_data):
    N = len(batch_data)
    drug_ids, protein_ids = [], []
    compound_max = 100
    protein_max = 1000
    compound_new = torch.zeros((N, compound_max), dtype=torch.long)
    protein_new = torch.zeros((N, protein_max), dtype=torch.long)
    labels_new = torch.zeros(N, dtype=torch.long)
    for i, pair in enumerate(batch_data):
        pair = pair.strip().split()
        drug_id, protein_id, compoundstr, proteinstr, label = pair[-5], pair[-4], pair[-3], pair[-2], pair[-1]
        drug_ids.append(drug_id)
        protein_ids.append(protein_id)
        compoundint = torch.from_numpy(label_smiles(
            compoundstr, CHARISOSMISET, compound_max))
        compound_new[i] = compoundint
        proteinint = torch.from_numpy(label_sequence(
            proteinstr, CHARPROTSET, protein_max))
        protein_new[i] = proteinint
        label = float(label)
        labels_new[i] = np.int16(label)
    return (compound_new, protein_new, labels_new)



MAX_SMI_LEN = 200
MAX_SEQ_LEN = 1500

# Extra tabular features from parquet (drug + protein level)
EXTRA_DRUG_FEATURE_COLS = ['molecular_weight']
EXTRA_PROTEIN_FEATURE_COLS = [
    'isoform_count', 'ft_transmem_count', 'pdb_count',
    'aac_A', 'aac_R', 'paac_APAAC1', 'paac_APAAC2',
]
EXTRA_PROTEIN_FEATURE_ALIASES = {
    'ctd_PolarizabilityD1075': ['ctd_PolarizabilityD1075', 'ctd__PolarizabilityD1075'],
    'ctd_PolarizabilityD1100': ['ctd_PolarizabilityD1100', 'ctd__PolarizabilityD1100'],
}

def resolve_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def get_extra_feature_columns(df):
    drug_cols = [c for c in EXTRA_DRUG_FEATURE_COLS if c in df.columns]
    prot_cols = [c for c in EXTRA_PROTEIN_FEATURE_COLS if c in df.columns]
    for canonical, aliases in EXTRA_PROTEIN_FEATURE_ALIASES.items():
        resolved = resolve_column(df, aliases)
        if resolved is not None:
            prot_cols.append(resolved)
    return drug_cols, prot_cols

print("Encoding helpers loaded")


Encoding helpers loaded


In [79]:
# Graph utility functions

def get_m2p_edge(mol_x, prot_x, mol_node_level=None):
    """Construct edges from atoms to amino acids"""
    x1 = np.arange(0, mol_x.shape[0])
    x1 = x1[mol_node_level==1] if mol_node_level is not None else x1
    x2 = np.arange(0, prot_x.shape[0])
    
    grid_x1, grid_x2 = np.meshgrid(x1, x2)
    edge_list = torch.LongTensor(np.vstack([grid_x1.ravel(), grid_x2.ravel()]))
    
    return edge_list

def maybe_num_nodes(index, num_nodes=None):
    return index.max().item() + 1 if num_nodes is None else int(num_nodes)

def get_self_loop_attr(edge_index, edge_attr, num_nodes):
    """Returns the edge features or weights of self-loops"""
    loop_mask = edge_index[0] == edge_index[1]
    loop_index = edge_index[0][loop_mask]

    if edge_attr is not None:
        loop_attr = edge_attr[loop_mask]
    else:
        loop_attr = torch.ones_like(loop_index, dtype=torch.float)

    num_nodes = maybe_num_nodes(edge_index, num_nodes)
    full_loop_attr = loop_attr.new_zeros((num_nodes, ) + loop_attr.size()[1:])
    full_loop_attr[loop_index] = loop_attr

    return full_loop_attr

print("Graph utility functions loaded")

Graph utility functions loaded


In [80]:
# MultiGraphData class

class MultiGraphData(Data):
    def __inc__(self, key, item, *args):
        if key == 'mol_edge_index':
            return self.mol_x.size(0)
        elif key == 'clique_edge_index':
            return self.clique_x.size(0)
        elif key == 'atom2clique_index':
            return torch.tensor([[self.mol_x.size(0)], [self.clique_x.size(0)]])
        elif key == 'prot_edge_index':
            return self.prot_node_aa.size(0)
        elif key == 'prot_struc_edge_index':
            return self.prot_node_aa.size(0)
        elif key == 'm2p_edge_index':
            return torch.tensor([[self.mol_x.size(0)], [self.prot_node_aa.size(0)]])
        else:
            return super(MultiGraphData, self).__inc__(key, item, *args)

print("MultiGraphData class loaded")

MultiGraphData class loaded


In [81]:
import torch.utils.data
from torch_geometric.data import Dataset
import torch
from torch_geometric.data import Data
import pickle
import torch.utils.data
import numpy as np
# label_sequence / label_smiles defined above


def get_m2p_edge(mol_x, prot_x, mol_node_level=None):
    """ Construct edges from atoms to amino acids """

    x1 = np.arange(0, mol_x.shape[0])
    x1 = x1[mol_node_level==1] if mol_node_level is not None else x1
    x2 = np.arange(0, prot_x.shape[0])
    
    grid_x1, grid_x2 = np.meshgrid(x1, x2)
    edge_list = torch.LongTensor(np.vstack([grid_x1.ravel(), grid_x2.ravel()]))
    
    return edge_list

class ProteinMoleculeDataset(Dataset):
    def __init__(self, sequence_data, mol_obj, prot_obj, device='cpu', drug_extra=None, prot_extra=None, drug_extra_dim=0, prot_extra_dim=0):
        super(ProteinMoleculeDataset, self).__init__()

        if isinstance(sequence_data,list):
            self.pairs = sequence_data
        else:
            raise Exception("provide list object")
        
        ## MOLECULES
        if isinstance(mol_obj, dict):
            self.mols = mol_obj
        elif isinstance(mol_obj, str):
            with open(mol_obj, 'rb') as f:
                self.mols = pickle.load(f)
        else:
            raise Exception("provide dict mol object or pickle path")


        ## PROTEINS
        if isinstance(prot_obj, dict):
            self.prots = prot_obj
        elif isinstance(prot_obj, str):
            self.prots = torch.load(prot_obj)
        else:
            raise Exception("provide dict mol object or pickle path")

        self.device = device
        self.drug_extra = drug_extra or {}
        self.prot_extra = prot_extra or {}
        self.drug_extra_dim = drug_extra_dim
        self.prot_extra_dim = prot_extra_dim

        for _, v in self.mols.items():
            v['atom_idx'] = v['atom_idx'].long().view(-1, 1)
            v['atom_feature'] = v['atom_feature'].float()
            adj = v['bond_feature'].long()
            mol_edge_index =  adj.nonzero(as_tuple=False).t().contiguous()
            v['atom_edge_index'] = mol_edge_index
            v['atom_edge_attr'] = adj[mol_edge_index[0], mol_edge_index[1]].long()
            v['atom_num_nodes'] = v['atom_idx'].shape[0]
            v['smiles_x'] = torch.tensor(label_smiles(v['smiles'], MAX_SMI_LEN=200)).reshape(1, -1)


        for _, v in self.prots.items():
            v['seq_feat'] = v['seq_feat'].float()
            v['token_representation'] = v['token_representation'].float()
            v['num_nodes'] = len(v['seq'])
            v['node_pos'] = torch.arange(len(v['seq'])).reshape(-1,1)
            v['edge_weight'] = v['edge_weight'].float()
            v['seq_x'] = torch.tensor(label_sequence(v['seq'], MAX_SEQ_LEN=1500)).reshape(1, -1)

    def get(self, index):
        return self.__getitem__(index)

    def len(self):
        return self.__len__()
    
    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # Extract data
        items = self.pairs[idx].split(' ')
        mol_key, prot_key = items[-3], items[-2]
        cls_y = torch.tensor(int(items[-1])).long()
        if mol_key not in self.mols or prot_key not in self.prots:
            raise KeyError(f'Missing graph cache for pair: {mol_key}, {prot_key}')
            
        mol = self.mols[mol_key]
        prot = self.prots[prot_key]
        
        ## atom
        mol_x = mol['atom_idx']
        mol_x_feat = mol['atom_feature']
        mol_edge_index  = mol['atom_edge_index']
        mol_edge_attr = mol['atom_edge_attr']
        mol_num_nodes = mol['atom_num_nodes']
        mol_smiles_x = mol['smiles_x']

        ## Prot
        prot_seq = prot['seq']
        prot_seq_x = prot['seq_x']
        prot_node_aa = prot['seq_feat']
        prot_node_evo = prot['token_representation']
        prot_num_nodes = prot['num_nodes']
        prot_node_pos = prot['node_pos']
        prot_edge_index = prot['edge_index']
        prot_edge_weight = prot['edge_weight']

        out = MultiGraphData(
                ## MOLECULE
                mol_x=mol_x, mol_smiles_x=mol_smiles_x, mol_x_feat=mol_x_feat, mol_edge_index=mol_edge_index,
                mol_edge_attr=mol_edge_attr, mol_num_nodes=mol_num_nodes,
                mol_node_levels=mol.get('node_levels'),
                ## PROTEIN
                prot_node_aa=prot_node_aa, prot_node_evo=prot_node_evo, prot_seq_x=prot_seq_x,
                prot_node_pos=prot_node_pos, prot_seq=prot_seq,
                prot_edge_index=prot_edge_index, prot_edge_weight=prot_edge_weight,
                prot_num_nodes=prot_num_nodes,
                ## Y output
                cls_y=cls_y,
                ## keys
                mol_key = mol_key, prot_key = prot_key,
                # BI GRAPH
                m2p_edge_index=get_m2p_edge(mol_x, prot_node_aa, mol.get('node_levels')),
                drug_extra_feat=self.drug_extra.get(
                    mol_key, torch.zeros(1, self.drug_extra_dim, dtype=torch.float)
                ),
                prot_extra_feat=self.prot_extra.get(
                    prot_key, torch.zeros(1, self.prot_extra_dim, dtype=torch.float)
                ),
        )

        return out

def maybe_num_nodes(index, num_nodes=None):
    return index.max().item() + 1 if num_nodes is None else int(num_nodes)

def get_self_loop_attr(edge_index, edge_attr, num_nodes):
    r"""Returns the edge features or weights of self-loops
    :math:`(i, i)` of every node :math:`i \in \mathcal{V}` in the
    graph given by :attr:`edge_index`. Edge features of missing self-loops not
    present in :attr:`edge_index` will be filled with zeros. If
    :attr:`edge_attr` is not given, it will be the vector of ones.

    .. note::
        This operation is analogous to getting the diagonal elements of the
        dense adjacency matrix.

    Args:
        edge_index (LongTensor): The edge indices.
        edge_attr (Tensor, optional): Edge weights or multi-dimensional edge
            features. (default: :obj:`None`)
        num_nodes (int, optional): The number of nodes, *i.e.*
            :obj:`max_val + 1` of :attr:`edge_index`. (default: :obj:`None`)

    :rtype: :class:`Tensor`

    Examples:

        >>> edge_index = torch.tensor([[0, 1, 0],
        ...                            [1, 0, 0]])
        >>> edge_weight = torch.tensor([0.2, 0.3, 0.5])
        >>> get_self_loop_attr(edge_index, edge_weight)
        tensor([0.5000, 0.0000])

        >>> get_self_loop_attr(edge_index, edge_weight, num_nodes=4)
        tensor([0.5000, 0.0000, 0.0000, 0.0000])
    """
    loop_mask = edge_index[0] == edge_index[1]
    loop_index = edge_index[0][loop_mask]

    if edge_attr is not None:
        loop_attr = edge_attr[loop_mask]
    else:  # A vector of ones:
        loop_attr = torch.ones_like(loop_index, dtype=torch.float)

    num_nodes = maybe_num_nodes(edge_index, num_nodes)
    full_loop_attr = loop_attr.new_zeros((num_nodes, ) + loop_attr.size()[1:])
    full_loop_attr[loop_index] = loop_attr

    return full_loop_attr



class MultiGraphData(Data):
    def __inc__(self, key, item, *args):
        if key == 'mol_edge_index':
            return self.mol_x.size(0)
        elif key == 'clique_edge_index':
            return self.clique_x.size(0)
        elif key == 'atom2clique_index':
            return torch.tensor([[self.mol_x.size(0)], [self.clique_x.size(0)]])
        elif key == 'prot_edge_index':
            return self.prot_node_aa.size(0)
        elif key == 'prot_struc_edge_index':
            return self.prot_node_aa.size(0)
        elif key == 'm2p_edge_index':
            return torch.tensor([[self.mol_x.size(0)], [self.prot_node_aa.size(0)]])
        else:
            return super(MultiGraphData, self).__inc__(key, item, *args)



In [82]:
# Loss functions

class PolyLoss(nn.Module):
    def __init__(self, weight_loss, DEVICE, epsilon=1.0):
        super(PolyLoss, self).__init__()
        self.CELoss = nn.CrossEntropyLoss(weight=weight_loss, reduction='none')
        self.epsilon = epsilon
        self.DEVICE = DEVICE

    def forward(self, predicted, labels):
        one_hot = torch.zeros((labels.shape[0], 2), device=self.DEVICE).scatter_(
            1, torch.unsqueeze(labels, dim=-1), 1)
        pt = torch.sum(one_hot * F.softmax(predicted, dim=1), dim=-1)
        ce = self.CELoss(predicted, labels)
        poly1 = ce + self.epsilon * (1-pt)
        return torch.mean(poly1)


class CELoss(nn.Module):
    def __init__(self, weight_CE, DEVICE):
        super(CELoss, self).__init__()
        self.DEVICE = DEVICE
        if weight_CE is not None:
            self.register_buffer('weight', weight_CE.float())
        else:
            self.weight = None

    def forward(self, predicted, labels):
        # Always fp32 CE (AMP gives fp16 logits; class weights are fp32)
        return F.cross_entropy(
            predicted.float(),
            labels.long(),
            weight=self.weight,
        )

print("Loss functions loaded")

Loss functions loaded


In [83]:
# -*- coding:utf-8 -*-
'''
Author: MrZQAQ
Date: 2022-03-26 17:04
LastEditTime: 2022-11-23 15:32
LastEditors: MrZQAQ
Description: Offer EarlyStoping function
FilePath: /MCANet/utils/EarlyStoping.py
'''

import numpy as np
import torch


class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""

    def __init__(self, savepath=None, patience=7, verbose=False, delta=0, num_n_fold=0):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement.
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = -np.inf
        self.early_stop = False
        self.delta = delta
        self.num_n_fold = num_n_fold
        self.savepath = savepath

    def __call__(self, score, model, num_epoch):

        if self.best_score == -np.inf:
            self.save_checkpoint(score, model, num_epoch)
            self.best_score = score

        elif score < self.best_score + self.delta:
            self.counter += 1
            print(
                f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.save_checkpoint(score, model, num_epoch)
            self.best_score = score
            self.counter = 0

    def save_checkpoint(self, score, model, num_epoch):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            print(
                f'Have a new best checkpoint: ({self.best_score:.6f} --> {score:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.savepath +
                   f'/valid_best_checkpoint-{model.device}.pth')


In [84]:
# -*- coding:utf-8 -*-
'''
Author: MrZQAQ
Date: 2022-03-29 14:00
LastEditTime: 2022-11-23 15:32
LastEditors: MrZQAQ
Description: Test model
FilePath: /MCANet/utils/TestModel.py
'''

import torch
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm
from prefetch_generator import BackgroundGenerator
from sklearn.metrics import (accuracy_score, average_precision_score, auc, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)


def test_precess(MODEL, pbar, LOSS, DEVICE, FOLD_NUM):
    if isinstance(MODEL, list):
        for item in MODEL:
            item.eval()
    else:
        MODEL.eval()
    test_losses = []
    Y, P, S = [], [], []
    with torch.no_grad():
        for i, data in pbar:
            '''data preparation '''
            compounds, proteins, labels = data
            compounds = compounds.to(DEVICE)
            proteins = proteins.to(DEVICE)
            labels = labels.to(DEVICE)

            if isinstance(MODEL, list):
                predicted_scores = torch.zeros(2).to(DEVICE)
                for i in range(len(MODEL)):
                    predicted_scores = predicted_scores + \
                        MODEL[i](compounds, proteins)
                predicted_scores = predicted_scores / FOLD_NUM
            else:
                predicted_scores = MODEL(compounds, proteins)
            loss = LOSS(predicted_scores, labels)
            correct_labels = labels.to('cpu').data.numpy()
            predicted_scores = F.softmax(
                predicted_scores, 1).to('cpu').data.numpy()
            predicted_labels = np.argmax(predicted_scores, axis=1)
            predicted_scores = predicted_scores[:, 1]

            Y.extend(correct_labels)
            P.extend(predicted_labels)
            S.extend(predicted_scores)
            test_losses.append(loss.item())
    Precision = precision_score(Y, P)
    Recall = recall_score(Y, P)
    if len(np.unique(Y)) < 2:
        AUC, PRC = float('nan'), float('nan')
    else:
        AUC = roc_auc_score(Y, S)
        PRC = average_precision_score(Y, S)
    Accuracy = accuracy_score(Y, P)
    test_loss = np.average(test_losses)
    return Y, P, test_loss, Accuracy, Precision, Recall, AUC, PRC


def test_MIF_precess(MODEL, pbar, LOSS, DEVICE, FOLD_NUM):
    if isinstance(MODEL, list):
        for item in MODEL:
            item.eval()
    else:
        MODEL.eval()
    test_losses = []
    Y, P, S = [], [], []
    with torch.no_grad():
        for i, data in pbar:
            '''data preparation '''
            data = data.to(DEVICE)
            if isinstance(MODEL, list):
                predicted_scores = torch.zeros(2).to(DEVICE)
                for i in range(len(MODEL)):
                    predicted_scores = predicted_scores + \
                        MODEL[i](data)
                predicted_scores = predicted_scores / FOLD_NUM
            else:
                predicted_scores = MODEL(data)
            labels = data.cls_y
            loss = LOSS(predicted_scores, labels)
            correct_labels = labels.to('cpu').data.numpy()
            predicted_scores = F.softmax(predicted_scores, 1).to('cpu').data.numpy()
            predicted_labels = np.argmax(predicted_scores, axis=1)
            predicted_scores = predicted_scores[:, 1]

            Y.extend(correct_labels)
            P.extend(predicted_labels)
            S.extend(predicted_scores)
            test_losses.append(loss.item())
    Precision = precision_score(Y, P)
    Recall = recall_score(Y, P)
    if len(np.unique(Y)) < 2:
        AUC, PRC = float('nan'), float('nan')
    else:
        AUC = roc_auc_score(Y, S)
        PRC = average_precision_score(Y, S)
    Accuracy = accuracy_score(Y, P)
    test_loss = np.average(test_losses)
    return Y, P, test_loss, Accuracy, Precision, Recall, AUC, PRC

def test_model(MODEL, dataset_loader, save_path, DATASET, LOSS, DEVICE, dataset_class="Train", save=True, FOLD_NUM=1, MIF=False):
    test_pbar = tqdm(
        enumerate(
            BackgroundGenerator(dataset_loader)),
        total=len(dataset_loader))
    #test_pbar = enumerate(BackgroundGenerator(dataset_loader))
    T, P, loss_test, Accuracy_test, Precision_test, Recall_test, AUC_test, PRC_test = \
        test_MIF_precess(MODEL, test_pbar, LOSS, DEVICE, FOLD_NUM) if MIF else test_precess(MODEL, test_pbar, LOSS, DEVICE, FOLD_NUM)
    if save:
        if FOLD_NUM == 1:
            filepath = save_path + \
                "/{}_{}_prediction.txt".format(DATASET, dataset_class)
        else:
            filepath = save_path + \
                "/{}_{}_ensemble_prediction.txt".format(DATASET, dataset_class)
        with open(filepath, 'a') as f:
            for i in range(len(T)):
                f.write(str(T[i]) + " " + str(P[i]) + '\n')
    results = '{}: Loss:{:.5f};Accuracy:{:.5f};Precision:{:.5f};Recall:{:.5f};AUC:{:.5f};PRC:{:.5f}.' \
        .format(dataset_class, loss_test, Accuracy_test, Precision_test, Recall_test, AUC_test, PRC_test)
    print(results)
    return results, Accuracy_test, Precision_test, Recall_test, AUC_test, PRC_test


In [85]:
from torch_geometric.nn import GATConv, SAGPooling, LayerNorm, global_add_pool

import torch
import torch.nn as nn
import torch.nn.functional as F


class MLP(nn.Module):

    def __init__(self, dims, out_norm=False, in_norm=False, bias=True): #L=nb_hidden_layers
        super().__init__()
        list_FC_layers = [ nn.Linear(dims[idx-1], dims[idx], bias=bias) for idx in range(1,len(dims)) ]
        self.FC_layers = nn.ModuleList(list_FC_layers)
        self.hidden_layers = len(dims) - 2

        self.out_norm = out_norm
        self.in_norm = in_norm

        if self.out_norm:
            self.out_ln = nn.LayerNorm(dims[-1])
        if self.in_norm:
            self.in_ln = nn.LayerNorm(dims[0])

    def reset_parameters(self):
        for idx in range(self.hidden_layers+1):
            self.FC_layers[idx].reset_parameters()
        if self.out_norm:
            self.out_ln.reset_parameters()
        if self.in_norm:
            self.in_ln.reset_parameters()

    def forward(self, x):
        y = x
        # Input Layer Norm
        if self.in_norm:
            y = self.in_ln(y)

        for idx in range(self.hidden_layers):
            y = self.FC_layers[idx](y)
            y = F.relu(y)
        y = self.FC_layers[-1](y)

        if self.out_norm:
            y = self.out_ln(y)

        return y


class CoAttentionLayer(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.n_features = n_features
        self.w_q = nn.Parameter(torch.zeros(n_features, n_features//2))
        self.w_k = nn.Parameter(torch.zeros(n_features, n_features//2))
        self.bias = nn.Parameter(torch.zeros(n_features // 2))
        self.a = nn.Parameter(torch.zeros(n_features//2))

        nn.init.xavier_uniform_(self.w_q)
        nn.init.xavier_uniform_(self.w_k)
        nn.init.xavier_uniform_(self.bias.view(*self.bias.shape, -1))
        nn.init.xavier_uniform_(self.a.view(*self.a.shape, -1))
    
    def forward(self, receiver, attendant):
        keys = receiver @ self.w_k
        queries = attendant @ self.w_q

        e_activations = queries.unsqueeze(-3) + keys.unsqueeze(-2) + self.bias
        e_scores = torch.tanh(e_activations) @ self.a
        attentions = e_scores
        return attentions
    

class RESCAL(nn.Module):

    def __init__(self, n_features, depth):
        super().__init__()
        self.n_features = n_features
        self.co_attn = CoAttentionLayer(n_features)
        self.mlp = nn.Sequential(
            nn.Linear(depth*depth, 2)
        )

    def forward(self, heads, tails):
        alpha_scores = self.co_attn(heads, tails)
        heads = F.normalize(heads, dim=-1)
        tails = F.normalize(tails, dim=-1)
        scores = (heads @ tails.transpose(-2, -1))
        scores *= alpha_scores
        scores = self.mlp(scores.reshape(scores.shape[0], -1))
        return scores
    
    def __repr__(self):
        return f"{self.__class__.__name__}({self.n_rels}, {self.rel_emb.weight.shape})"
    
class PoolAttention(nn.Module):
    """利用Attention进行多模态融合, `with-attn`变体的关键组件"""

    def __init__(self, n_features, num_neads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(n_features, num_heads=num_neads, batch_first=True)
        self.drug_norm = nn.LayerNorm(n_features)
        self.prot_norm = nn.LayerNorm(n_features)
        self.mlp = nn.Sequential(
            nn.Linear(n_features*2, n_features*2),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(n_features*2, n_features*1),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(n_features*1, 2),
        )

    def forward(self, drug, prot):
        drug = self.drug_norm(drug)
        prot = self.prot_norm(prot)
        drug_attn = self.attn(drug, prot, prot)[0]
        prot_attn = self.attn(prot, drug, drug)[0]
        drug_pool = torch.max((drug+drug_attn)/2, dim=1)[0]
        prot_pool = torch.max((prot+prot_attn)/2, dim=1)[0]
        scores = self.mlp(torch.cat([drug_pool, prot_pool], dim=-1))
        return scores

class AttentionLayer(nn.Module):
    def __init__(self, n_features, heads=4):
        super().__init__()
        self.n_features = n_features
        self.heads = heads
        self.attn = nn.MultiheadAttention(self.n_features, self.heads, batch_first=True, dropout=0.3)
        self.mlp = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(self.n_features*2, self.n_features),
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.n_features, self.n_features),
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.n_features, 2)
        )

    def forward(self, drug_repr, prot_repr):
        drug_output, _ = self.attn(drug_repr, prot_repr, prot_repr)
        prot_output, _ = self.attn(prot_repr, drug_repr, drug_repr)

        drug_output = drug_output * 0.5 + drug_repr * 0.5
        prot_output = prot_output * 0.5 + prot_repr * 0.5

        drug_pool, _ = torch.max(drug_output, dim=1)
        prot_pool, _ = torch.max(prot_output, dim=1)
        concat_repr = torch.cat([drug_pool, prot_pool], -1)
        result = self.mlp(concat_repr)
        return result
    

def rbf(D, D_min=0., D_max=1., D_count=16, device='cpu'):
    '''
    From https://github.com/jingraham/neurips19-graph-protein-design

    Returns an RBF embedding of `torch.Tensor` `D` along a new axis=-1.
    That is, if `D` has shape [...dims], then the returned tensor will have
    shape [...dims, D_count].
    '''
    D = torch.where(D < D_max, D, torch.tensor(D_max).float().to(device) )
    D_mu = torch.linspace(D_min, D_max, D_count, device=device)
    D_mu = D_mu.view([1, -1])
    D_sigma = (D_max - D_min) / D_count
    D_expand = torch.unsqueeze(D, -1)

    RBF = torch.exp(-((D_expand - D_mu) / D_sigma) ** 2)
    return RBF

def get_CNNs(input_dim, conv_dim, kernel):
    return nn.Sequential(
            nn.Conv1d(in_channels=input_dim, out_channels=conv_dim, kernel_size=kernel[0]),
            nn.ReLU(),
            nn.Conv1d(in_channels=conv_dim, out_channels=conv_dim*2, kernel_size=kernel[1]),
            nn.ReLU(),
            nn.Conv1d(in_channels=conv_dim*2, out_channels=conv_dim*4, kernel_size=kernel[2]),
            nn.ReLU(),
        )

# -*- coding:utf-8 -*-

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Embedding
# layers defined above
from torch_geometric.nn import (
                                GATConv,
                                SAGPooling,
                                LayerNorm,
                                global_add_pool
                                )


class MIF_conv_block(nn.Module):
    def __init__(self, in_channels=200, out_channels=200, num_heads=4, dropout=0.3):
        super(MIF_conv_block, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_heads = num_heads
        self.dropout = dropout

        self.conv = GATConv(self.in_channels, self.out_channels//self.num_heads, self.num_heads, dropout=self.dropout)
        self.norm = LayerNorm(self.in_channels)
        self.readout = SAGPooling(self.out_channels, min_score=-1)

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = F.elu(self.norm(x, batch))
        x = self.conv(x, edge_index, edge_attr)
        x, _, _, x_batch, _, _ = self.readout(x, edge_index, edge_attr=edge_attr, batch=batch)
        global_graph_emb = global_add_pool(x, x_batch)
        return x, global_graph_emb


class MIFBlock(nn.Module):
    def __init__(self, in_channels=200, out_channels=200, num_heads=5, dropout=0.4):
        super(MIFBlock, self).__init__()
        
        self.hidden_channels = out_channels // (num_heads*2)
        self.drug_conv = GATConv(in_channels, self.hidden_channels, num_heads, dropout=0.1)
        self.prot_conv = GATConv(in_channels, self.hidden_channels, num_heads, dropout=0.3)
        self.inter_conv = GATConv((in_channels, in_channels), self.hidden_channels, num_heads, dropout=dropout)
        self.drug_norm = LayerNorm(out_channels)
        self.prot_norm = LayerNorm(out_channels)
        self.drug_pool = GATConv(out_channels, out_channels//num_heads, num_heads)
        self.prot_pool = SAGPooling(out_channels, min_score=-1)
        # self.prot_pool = GATConv(out_channels, out_channels//num_heads, num_heads)

    def forward(self, atom_x, atom_edge_index, bond_x, atom_batch, \
                aa_x, aa_edge_index, aa_edge_attr, aa_batch, m2p_edge_index):
        
        atom_x_res = atom_x
        aa_x_res = aa_x

        atom_intra_x = self.drug_conv(atom_x, atom_edge_index, bond_x)
        atom_inter_x = self.inter_conv((aa_x, atom_x), m2p_edge_index[[1,0]])
        atom_x_tmp = torch.cat([atom_intra_x, atom_inter_x], -1)
        atom_x = F.elu(self.drug_norm(atom_x_tmp, atom_batch))

        aa_intra_x = self.prot_conv(aa_x, aa_edge_index, aa_edge_attr)
        aa_inter_x = self.inter_conv((atom_x, aa_x), m2p_edge_index)
        aa_x_tmp = torch.cat([aa_intra_x, aa_inter_x], -1)
        aa_x = F.elu(self.prot_norm(aa_x_tmp, aa_batch))

        atom_x = self.drug_pool(atom_x, atom_edge_index, bond_x)
        aa_x, _, _, aa_batch, _, _ = self.prot_pool(aa_x, aa_edge_index, edge_attr=aa_edge_attr, batch=aa_batch)
        # aa_x, aa_edge_index, aa_edge_attr, aa_batch, _, _ = self.prot_pool(aa_x, aa_edge_index, edge_attr=aa_edge_attr, batch=aa_batch)
        # aa_x = self.prot_pool(aa_x, aa_edge_index, aa_edge_attr)
        atom_x = F.dropout(atom_x_res+F.elu(atom_x), 0.1, self.training)
        aa_x = F.dropout(aa_x_res+F.elu(aa_x), 0.1, self.training)
        drug_global_repr = global_add_pool(atom_x, atom_batch)
        prot_global_repr = global_add_pool(aa_x, aa_batch)

        return atom_x, aa_x, drug_global_repr, prot_global_repr

class MIFBlock_1D(nn.Module):
    def __init__(self, input_dim=200, conv=50, drug_kernel=[4, 6, 8], prot_kernel=[4, 8, 12]):
        super(MIFBlock_1D, self).__init__()
        self.attention_dim = conv * 4
        self.mix_attention_head = 5

        self.Drug_CNNs = get_CNNs(input_dim, conv, drug_kernel)
        self.Protein_CNNs = get_CNNs(input_dim, conv, prot_kernel)

        self.mix_attention_layer = nn.MultiheadAttention(self.attention_dim, self.mix_attention_head, batch_first=True, dropout=0.3)

    def forward(self, drugembed, proteinembed):

        # [batch_size, seq_len, embed_dim] -> [batch_size, embed_dim, seq_len] 
        drugembed = drugembed.permute(0, 2, 1)
        proteinembed = proteinembed.permute(0, 2, 1)

        drugConv = self.Drug_CNNs(drugembed)
        proteinConv = self.Protein_CNNs(proteinembed)

        # [batch_size, embed_dim, seq_len] -> [batch_size, seq_len, embed_dim]
        drugConv = drugConv.permute(0, 2, 1)
        proteinConv = proteinConv.permute(0, 2, 1)

        # cross Attention
        drug_att, _ = self.mix_attention_layer(drugConv, proteinConv, proteinConv)
        protein_att, _ = self.mix_attention_layer(proteinConv, drugConv, drugConv)

        drugConv = drugConv * 0.5 + drug_att * 0.5
        proteinConv = proteinConv * 0.5 + protein_att * 0.5

        drugPool, _ = torch.max(drugConv, dim=1)
        proteinPool, _ = torch.max(proteinConv, dim=1)

        return drugConv, proteinConv, drugPool, proteinPool


class MIFDTI(nn.Module):
    def __init__(self, depth=3, device='cuda:0', drug_extra_dim=0, prot_extra_dim=0):
        super(MIFDTI, self).__init__()

        self.drug_in_channels = 43
        self.prot_in_channels = 33
        self.prot_evo_in_channels = 1280
        self.hidden_channels = 200
        self.depth = depth
        self.device = device

        # MOLECULE IN FEAT
        self.atom_type_encoder = Embedding(20, self.hidden_channels)
        self.atom_feat_encoder = MLP([self.drug_in_channels, self.hidden_channels * 2, self.hidden_channels], out_norm=True) 
        self.bond_encoder = Embedding(10, self.hidden_channels)

        # PROTEIN IN FEAT
        self.prot_evo = MLP([self.prot_evo_in_channels, self.hidden_channels * 2, self.hidden_channels], out_norm=True) 
        self.prot_aa = MLP([self.prot_in_channels, self.hidden_channels * 2, self.hidden_channels], out_norm=True) 

        # ENCODER
        self.blocks = nn.ModuleList([MIFBlock() for _ in range(depth)])

        self.drug_seq_emb = nn.Embedding(65, self.hidden_channels, padding_idx=0)
        self.prot_seq_emb = nn.Embedding(26, self.hidden_channels, padding_idx=0)
        self.blocks_1D = nn.ModuleList([MIFBlock_1D() for _ in range(depth)])

        self.attn = RESCAL(self.hidden_channels, self.depth*2)
        self.drug_extra_dim = drug_extra_dim
        self.prot_extra_dim = prot_extra_dim
        if drug_extra_dim > 0:
            self.drug_extra_encoder = MLP([drug_extra_dim, self.hidden_channels, self.hidden_channels], out_norm=True)
        else:
            self.drug_extra_encoder = None
        if prot_extra_dim > 0:
            self.prot_extra_encoder = MLP([prot_extra_dim, self.hidden_channels, self.hidden_channels], out_norm=True)
        else:
            self.prot_extra_encoder = None

        self.to(device)

    def forward(self,data):

        # Molecule
        atom_x, atom_x_feat, smiles_x, atom_edge_index, bond_x, mol_node_levels = \
            data.mol_x, data.mol_x_feat, data.mol_smiles_x, data.mol_edge_index, data.mol_edge_attr, data.mol_node_levels
        # Protein (amino acid)
        aa_x, aa_evo_x, seq_x, aa_edge_index, aa_edge_weight = \
            data.prot_node_aa, data.prot_node_evo, data.prot_seq_x, data.prot_edge_index, data.prot_edge_weight, \
        # Batch
        atom_batch, aa_batch = data.mol_x_batch, data.prot_node_aa_batch
        # Bi Graph
        m2p_edge_index = data.m2p_edge_index

        # MOLECULE Featurize
        atom_x = self.atom_type_encoder(atom_x.squeeze()) + self.atom_feat_encoder(atom_x_feat)
        bond_x = self.bond_encoder(bond_x)
                
        # PROTEIN Featurize
        aa_x = self.prot_aa(aa_x) + self.prot_evo(aa_evo_x)
        aa_edge_attr = rbf(aa_edge_weight, D_max=1.0, D_count=self.hidden_channels, device=self.device)

        # Encoding
        drug_repr = []
        prot_repr = []
        for i in range(self.depth):
            out = self.blocks[i](atom_x, atom_edge_index, bond_x, atom_batch, \
                                 aa_x, aa_edge_index, aa_edge_attr, aa_batch, \
                                 m2p_edge_index)
            atom_x, aa_x, drug_global_repr, prot_global_repr = out
            drug_global_repr = atom_x[mol_node_levels==2]
            drug_repr.append(drug_global_repr)
            prot_repr.append(prot_global_repr)

        atom_x_seq = self.drug_seq_emb(smiles_x)
        aa_x_seq = self.prot_seq_emb(seq_x)
        for i in range(self.depth):
            out_seq = self.blocks_1D[i](atom_x_seq, aa_x_seq)
            atom_x_seq, aa_x_seq, drug_seq_pool, prot_seq_pool = out_seq
            drug_repr.append(drug_seq_pool)
            prot_repr.append(prot_seq_pool)

        drug_repr = torch.stack(drug_repr, dim=-2)
        prot_repr = torch.stack(prot_repr, dim=-2)

        if self.drug_extra_encoder is not None and hasattr(data, 'drug_extra_feat'):
            drug_extra = self.drug_extra_encoder(data.drug_extra_feat.to(self.device).float())
            drug_repr = drug_repr + drug_extra.unsqueeze(1)
        if self.prot_extra_encoder is not None and hasattr(data, 'prot_extra_feat'):
            prot_extra = self.prot_extra_encoder(data.prot_extra_feat.to(self.device).float())
            prot_repr = prot_repr + prot_extra.unsqueeze(1)

        scores = self.attn(drug_repr, prot_repr)

        return scores

def get_m2p_edge_from_batch(atom_batch, aa_batch, node_level=None):

    mask = atom_batch.unsqueeze(1) == aa_batch.unsqueeze(0)  # (num_a_nodes, num_b_nodes) 的bool矩阵
    if node_level is not None:
        mask = mask * (node_level==1).unsqueeze(1)
    a_idx, b_idx = torch.nonzero(mask, as_tuple=True)
    edge_list = torch.stack([a_idx, b_idx], dim=0)
    return edge_list
print('MIF-DTI model loaded')


MIF-DTI model loaded


In [86]:
# Debug: inspect batch structure
# sample_batch = next(iter(train_loader))
# sample_batch = sample_batch.to(DEVICE)

# print("Batch attributes:")
# for attr in dir(sample_batch):
#     if not attr.startswith('_'):
#         try:
#             val = getattr(sample_batch, attr)
#             if isinstance(val, torch.Tensor):
#                 print(f"  {attr}: {val.shape}")
#         except:
#             pass

# print("\n_slice_dict:")
# if hasattr(sample_batch, '_slice_dict'):
#     for key, val in sample_batch._slice_dict.items():
#         print(f"  {key}: {val}")

# print("\n_inc_dict:")
# if hasattr(sample_batch, '_inc_dict'):
#     for key, val in sample_batch._inc_dict.items():
#         print(f"  {key}: {val}")

In [87]:
# Load parquet data and prepare for training
import pandas as pd
from sklearn.model_selection import train_test_split

if os.environ.get("MIF_DTI_ENV") == "local_conda":
    DATA_DIR = globals().get("LOCAL_DATA_DIR", os.path.join(os.getcwd(), "data"))
    SAVE_DIR = globals().get("LOCAL_SAVE_DIR", os.path.join(os.getcwd(), "dti_run"))
else:
    DATA_DIR = "/kaggle/input/datasets/gazimaksudurrahman/dti-data/data"
    if not os.path.exists(DATA_DIR):
        DATA_DIR = "data"
    SAVE_DIR = "/kaggle/working/dti_run"

DATASET = "KIBA"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DATA_DIR={DATA_DIR}")
print(f"SAVE_DIR={SAVE_DIR}")

os.makedirs(SAVE_DIR, exist_ok=True)

# Load dataset - using KIBA for this example
print("Loading KIBA dataset...")
df = pd.read_parquet(f"{DATA_DIR}/kiba.parquet")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"\nAffinity label distribution:")
print(df['affinity_label'].describe())

# Convert to binary classification (binding vs non-binding)
# KIBA values: lower = stronger binding. Common threshold is 12.1
THRESHOLD = 12.1
df['binary_label'] = (df['affinity_label'] < THRESHOLD).astype(int)

print(f"\nBinary label distribution:")
print(df['binary_label'].value_counts())

# Check for missing values
print(f"\nMissing SMILES: {df['drug_smiles'].isna().sum()}")
print(f"Missing sequences: {df['target_sequence'].isna().sum()}")

# Drop rows with missing data
df = df.dropna(subset=['drug_smiles', 'target_sequence'])
print(f"\nDataset after dropping NaN: {df.shape}")


DATA_DIR=d:\Thesis\DTI_Models\Runners\MIF-DTI\data
SAVE_DIR=d:\Thesis\DTI_Models\Runners\MIF-DTI\dti_run
Loading KIBA dataset...
Dataset shape: (117657, 157)
Columns: 157

Affinity label distribution:
count    117657.000000
mean         11.720685
std           0.834272
min           0.000000
25%          11.200000
50%          11.520216
75%          11.923909
max          17.200179
Name: affinity_label, dtype: float64

Binary label distribution:
1    92998
0    24659
Name: binary_label, dtype: int64

Missing SMILES: 0
Missing sequences: 0

Dataset after dropping NaN: (117657, 158)


In [88]:
# Split data into train/val/test
from sklearn.model_selection import train_test_split

# First split: 80% train+val, 20% test
train_val_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['binary_label']
)

# Second split: 75% train, 25% val (of the train_val set)
train_df, val_df = train_test_split(
    train_val_df, 
    test_size=0.25, 
    random_state=42, 
    stratify=train_val_df['binary_label']
)

print(f"Train set: {len(train_df)} samples")
print(f"Val set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")

print(f"\nTrain label distribution:\n{train_df['binary_label'].value_counts()}")
print(f"\nVal label distribution:\n{val_df['binary_label'].value_counts()}")
print(f"\nTest label distribution:\n{test_df['binary_label'].value_counts()}")

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

Train set: 70593 samples
Val set: 23532 samples
Test set: 23532 samples

Train label distribution:
1    55798
0    14795
Name: binary_label, dtype: int64

Val label distribution:
1    18600
0     4932
Name: binary_label, dtype: int64

Test label distribution:
1    18600
0     4932
Name: binary_label, dtype: int64


In [89]:
from rdkit import Chem, RDLogger
from rdkit.Chem import BRICS

# Suppress RDKit deprecation chatter during bulk featurization (not errors)
RDLogger.DisableLog('rdApp.warning')


def _implicit_valence(atom):
    """RDKit 2025+ prefers GetValence(ValenceType.IMPLICIT) over GetImplicitValence()."""
    try:
        return atom.GetValence(Chem.ValenceType.IMPLICIT)
    except AttributeError:
        return atom.GetImplicitValence()
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from collections import defaultdict

def get_mol(smiles):
    return Chem.MolFromSmiles(smiles)

def get_smiles(mol):
    return Chem.MolToSmiles(mol, kekuleSmiles=True)

def copy_atom(atom):
    new_atom = Chem.Atom(atom.GetSymbol())
    new_atom.SetFormalCharge(atom.GetFormalCharge())
    new_atom.SetAtomMapNum(atom.GetAtomMapNum())
    return new_atom

def copy_edit_mol(mol):
    new_mol = Chem.RWMol(Chem.MolFromSmiles(''))
    for atom in mol.GetAtoms():
        new_atom = copy_atom(atom)
        new_mol.AddAtom(new_atom)
    for bond in mol.GetBonds():
        a1 = bond.GetBeginAtom().GetIdx()
        a2 = bond.GetEndAtom().GetIdx()
        bt = bond.GetBondType()
        new_mol.AddBond(a1, a2, bt)
    return new_mol

def get_clique_mol(mol, atoms):
    # get the fragment of clique
    Chem.Kekulize(mol)
    smiles = Chem.MolFragmentToSmiles(mol, atoms, kekuleSmiles=True)
    new_mol = Chem.MolFromSmiles(smiles, sanitize=False)
    new_mol = copy_edit_mol(new_mol).GetMol()
    new_mol = sanitize(new_mol)  # We assume this is not None
    Chem.SanitizeMol(mol)
    return new_mol

def sanitize(mol):
    try:
        smiles = get_smiles(mol)
        mol = get_mol(smiles)
    except Exception as e:
        return None
    return mol

def motif_decomp(mol):
    n_atoms = mol.GetNumAtoms()
    if n_atoms == 1:
        return [[0]]

    cliques = []  
    for bond in mol.GetBonds():
        a1 = bond.GetBeginAtom().GetIdx()
        a2 = bond.GetEndAtom().GetIdx()
        cliques.append([a1, a2])  

    res = list(BRICS.FindBRICSBonds(mol))  
    if len(res) != 0:
        for bond in res:
            if [bond[0][0], bond[0][1]] in cliques:
                cliques.remove([bond[0][0], bond[0][1]])
            else:
                cliques.remove([bond[0][1], bond[0][0]])
            cliques.append([bond[0][0]])
            cliques.append([bond[0][1]]) 

    for c in range(len(cliques) - 1):
        if c >= len(cliques):
            break
        for k in range(c + 1, len(cliques)):
            if k >= len(cliques):
                break
            if len(set(cliques[c]) & set(cliques[k])) > 0: 
                cliques[c] = list(set(cliques[c]) | set(cliques[k]))
                cliques[k] = []
        cliques = [c for c in cliques if len(c) > 0]
    cliques = [c for c in cliques if n_atoms> len(c) > 0]


    num_cli = len(cliques)
    ssr_mol = Chem.GetSymmSSSR(mol)
    for i in range(num_cli):
        c = cliques[i]
        cmol = get_clique_mol(mol, c)
        ssr = Chem.GetSymmSSSR(cmol)
        if len(ssr)>1: 
            for ring in ssr_mol:
                if len(set(list(ring)) & set(c)) == len(list(ring)):
                    cliques.append(list(ring))
            cliques[i]=[]
    
    cliques = [c for c in cliques if n_atoms> len(c) > 0]
    return cliques

from rdkit import Chem
from rdkit.Chem.rdchem import BondType

from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig
import os
import numpy as np
# motif_decomp defined above

import torch

fdefName = os.path.join(RDConfig.RDDataDir,'BaseFeatures.fdef')
factory = ChemicalFeatures.BuildFeatureFactory(fdefName)
import sys

from tqdm import tqdm




def one_of_k_encoding(x, allowable_set):
    if x not in allowable_set:
        raise Exception("input {0} not in allowable set{1}:".format(x, allowable_set))
    return list(map(lambda s: x == s, allowable_set))

def one_of_k_encoding_unk(x, allowable_set):
    """Maps inputs not in the allowable set to the last element."""
    if x not in allowable_set:
        x = allowable_set[-1]
    return list(map(lambda s: x == s, allowable_set))


def atom_features(atom):
    #encoding = one_of_k_encoding_unk(atom.GetSymbol(),['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na','Ca', 'Fe', 'As', 'Al', 'I', 'B', 'V', 'K', 'Tl', 'Yb','Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti', 'Zn', 'H','Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr','Cr', 'Pt', 'Hg', 'Pb', 'Unknown'])
    encoding = one_of_k_encoding(atom.GetDegree(), [0,1,2,3,4,5,6,7,8,9,10]) + one_of_k_encoding_unk(atom.GetTotalNumHs(), [0,1,2,3,4,5,6,7,8,9,10])
    encoding += one_of_k_encoding_unk(_implicit_valence(atom), [0,1,2,3,4,5,6,7,8,9,10]) 
    encoding += one_of_k_encoding_unk(atom.GetHybridization(), [
                      Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
                      Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
                      Chem.rdchem.HybridizationType.SP3D2, 'other'])
    # encoding += one_of_k_encoding_unk(atom.GetFormalCharge(), [0,-1,1,2,-100]) 
    # encoding += one_of_k_encoding_unk(atom.GetNumRadicalElectrons(), [0,1,2,-100]) 
    encoding += [atom.GetIsAromatic()]
    # encoding += [atom.IsInRing()]

    try:
        encoding += one_of_k_encoding_unk(
                    atom.GetProp('_CIPCode'),
                    ['R', 'S']) + [atom.HasProp('_ChiralityPossible')]
    except:
        encoding += [0, 0] + [atom.HasProp('_ChiralityPossible')]
    
    return np.array(encoding)



class MoleculeGraphDataset():
    def __init__(self,atom_classes=None, halogen_detail=False, save_path=None):
        ## ATOM CLASSES ##
        self.ATOM_CODES = {}
        if atom_classes is None:
            metals = ([3, 4, 11, 12, 13] + list(range(19, 32))
                      + list(range(37, 51)) + list(range(55, 84))
                      + list(range(87, 104)))

            self.FEATURE_NAMES = []
            if halogen_detail:
                atom_classes = [
                    (5, 'B'),
                    (6, 'C'),
                    (7, 'N'),
                    (8, 'O'),
                    (15, 'P'),
                    (16, 'S'),
                    (34, 'Se'),
                    ## halogen
                    (9, 'F'),
                    (17, 'Cl'),
                    (35, 'Br'),
                    (53, 'I'),
                    ## halogen
                    (metals, 'metal')
                ]
            else:
                atom_classes = [
                    (5, 'B'),
                    (6, 'C'),
                    (7, 'N'),
                    (8, 'O'),
                    (15, 'P'),
                    (16, 'S'),
                    (34, 'Se'),
                    ## halogen
                    ([9, 17, 35, 53], 'halogen'),
                    ## halogen
                    (metals, 'metal')
                ]
            

        self.NUM_ATOM_CLASSES = len(atom_classes)
        for code, (atom, name) in enumerate(atom_classes):
            if type(atom) is list:
                for a in atom:
                    self.ATOM_CODES[a] = code
            else:
                self.ATOM_CODES[atom] = code
            self.FEATURE_NAMES.append(name)

        ## Extra atom feature to extract
        self.feat_types = ['Donor', 'Acceptor', 'Hydrophobe', 'LumpedHydrophobe']

        ## Bond feature
        self.edge_dict = {BondType.SINGLE: 1, BondType.DOUBLE: 2,
                     BondType.TRIPLE: 3, BondType.AROMATIC: 4,
                         BondType.UNSPECIFIED: 1}
        ## File Paths
        self.save_path = save_path

    def hybridization_onehot(self,hybrid_type):
        hybrid_type = str(hybrid_type)
        types = {'S': 0, 'SP': 1, 'SP2': 2, 'SP3': 3, 'SP3D': 4, 'SP3D2': 5}

        encoding = np.zeros(len(types))
        try:
            encoding[types[hybrid_type]] = 1.0
        except:
            pass
        return encoding

    def encode_num(self,atomic_num):
        """Encode atom type with a binary vector. If atom type is not included in
        the `atom_classes`, its encoding is an all-zeros vector.

        Parameters
        ----------
        atomic_num: int
            Atomic number

        Returns
        -------
        encoding: np.ndarray
            Binary vector encoding atom type (one-hot or null).
        """

        if not isinstance(atomic_num, int):
            raise TypeError('Atomic number must be int, %s was given'
                            % type(atomic_num))

        encoding = np.zeros(self.NUM_ATOM_CLASSES)
        try:
            encoding[self.ATOM_CODES[atomic_num]] = 1.0
        except:
            pass
        return encoding

    def atom_feature_extract(self,atom):
        '''
            Atom Feature Extraction:
                0 - Degree
                1 - Total Valency
                2 to 7 - Hybridization Type One-hot Encoding
                8 - Number of Radical Electrons
                9 - Number of Formal Charge
                10 - Aromatic
                11 - Belongs to a Ring
                12 - Final to X belongs to Atom Classes
        '''

        feat = []

        feat.append(atom.GetDegree())
        feat.append(atom.GetTotalValence())
        feat += self.hybridization_onehot(atom.GetHybridization()).tolist()
        feat.append(atom.GetNumRadicalElectrons())
        feat.append(atom.GetFormalCharge())
        feat.append(int(atom.GetIsAromatic()))
        feat.append(int(atom.IsInRing()))
        # Atom class
        #feat += self.encode_num(atom.GetAtomicNum()).tolist()

        return feat

    def mol_feature(self,mol):
        atom_ids = []
        atom_feats = []
        for atom in mol.GetAtoms():
            atom_ids.append(atom.GetIdx())
            feat = self.atom_feature_extract(atom)
            atom_feats.append(feat)

        feature = np.array(list(zip(*sorted(zip(atom_ids, atom_feats))))[-1])

        return feature

    def mol_extra_feature(self, mol):
        atom_num = len(mol.GetAtoms())
        feature = np.zeros((atom_num, len(self.feat_types)))

        fact_feats = factory.GetFeaturesForMol(mol)
        for f in fact_feats:
            f_type = f.GetFamily()
            if f_type in self.feat_types:
                f_index = self.feat_types.index(f_type)
                atom_ids = f.GetAtomIds()
                feature[atom_ids, f_index] = 1

        return feature

    def mol_simplified_feature(self,mol):
        atom_ids = []
        atom_feats = []
        for atom in mol.GetAtoms():
            atom_ids.append(atom.GetIdx())
            atomic_num = atom.GetAtomicNum()

            if atomic_num in self.ATOM_CODES.keys():
                atom_feats.append([self.ATOM_CODES[atomic_num] + 1])
            else:
                atom_feats.append([0])
        feature = np.array(list(zip(*sorted(zip(atom_ids, atom_feats))))[-1])

        return feature
    
    def mol_sequence_simplified_feature(self,mol):
        
        atom_ids = []
        atom_feats = []
        for atom in mol.GetAtoms():
        
            atom_ids.append(atom.GetIdx())
            onehot_label = one_of_k_encoding_unk(atom.GetSymbol(),
                                  ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe', 'As', 'Al',
                                   'I', 'B', 'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti', 'Zn', 'H',
                                   'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb',
                                   'Unknown'])
            out = np.array(onehot_label).nonzero()[0]
            atom_feats.append(out)
        
        feature = np.array(list(zip(*sorted(zip(atom_ids, atom_feats))))[-1])
        
        return feature




    def mol_full_feature(self, mol):
        atom_ids = []
        atom_feats = []
        for atom in mol.GetAtoms():
            atom_ids.append(atom.GetIdx())
            feature = atom_features(atom)
            atom_feats.append(feature)
        feature = np.array(list(zip(*sorted(zip(atom_ids, atom_feats))))[-1])

        return feature

    def bond_feature(self,mol):
        atom_num = len(mol.GetAtoms())
        adj = np.zeros((atom_num,atom_num))

        for b in mol.GetBonds():
            v1 = b.GetBeginAtomIdx()
            v2 = b.GetEndAtomIdx()
            b_type = self.edge_dict[b.GetBondType()]
            adj[v1 - 1, v2 - 1] = b_type
            adj[v2 - 1, v1 - 1] = b_type

        return adj

    def junction_tree(self,mol):
        tree_edge_index, atom2clique_index, num_cliques, x_clique = tree_decomposition(mol,return_vocab=True)
        ## if weird compounds => each assign the separate cluster
        if atom2clique_index.nelement() == 0:
            num_cliques = len(mol.GetAtoms())
            x_clique = torch.tensor([3]*num_cliques)
            atom2clique_index = torch.stack([torch.arange(num_cliques),
                                             torch.arange(num_cliques)])
        tree = dict(tree_edge_index=tree_edge_index,
             atom2clique_index=atom2clique_index,
             num_cliques=num_cliques,
             x_clique=x_clique)
        
        return tree


    def featurize(self,mol,type='atom_type'):
        if type=='atom_type':
            atom_feature = self.mol_simplified_feature(mol)
        elif type =='detailed_atom_type':
            atom_feature = self.mol_sequence_simplified_feature(mol)
        elif type=='atom_feature':
            base_feat = self.mol_feature(mol)
            extra_feat = self.mol_extra_feature(mol)
            atom_feature = np.concatenate((base_feat, extra_feat), axis=1)
           
        elif type=='atom_full_feature':
            atom_feature = self.mol_full_feature(mol)
            #extra_feat = self.mol_extra_feature(mol)
            #atom_feature = np.concatenate((base_feat, extra_feat), axis=1)
        else:
            raise Exception('atom_type or atom_feature')
        bond_feature = self.bond_feature(mol)

        return atom_feature, bond_feature




## FILE from pytorch geometric version 2.
from itertools import chain
from typing import Any, Tuple, Union

import torch
from scipy.sparse.csgraph import minimum_spanning_tree
from torch import Tensor

from torch_geometric.utils import (
    from_scipy_sparse_matrix,
    to_scipy_sparse_matrix,
    to_undirected,
)


def tree_decomposition(
    mol: Any,
    return_vocab: bool = False,
) -> Union[Tuple[Tensor, Tensor, int], Tuple[Tensor, Tensor, int, Tensor]]:
    r"""The tree decomposition algorithm of molecules from the
    `"Junction Tree Variational Autoencoder for Molecular Graph Generation"
    <https://arxiv.org/abs/1802.04364>`_ paper.
    Returns the graph connectivity of the junction tree, the assignment
    mapping of each atom to the clique in the junction tree, and the number
    of cliques.

    Args:
        mol (rdkit.Chem.Mol): An :obj:`rdkit` molecule.
        return_vocab (bool, optional): If set to :obj:`True`, will return an
            identifier for each clique (ring, bond, bridged compounds, single).
            (default: :obj:`False`)

    :rtype: :obj:`(LongTensor, LongTensor, int)` if :obj:`return_vocab` is
        :obj:`False`, else :obj:`(LongTensor, LongTensor, int, LongTensor)`
    """
    import rdkit.Chem as Chem

    # Cliques = rings and bonds.
    cliques = [list(x) for x in Chem.GetSymmSSSR(mol)]
    xs = [0] * len(cliques)
    for bond in mol.GetBonds():
        if not bond.IsInRing():
            cliques.append([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])
            xs.append(1)

    # Generate `atom2clique` mappings.
    atom2clique = [[] for i in range(mol.GetNumAtoms())]
    for c in range(len(cliques)):
        for atom in cliques[c]:
            atom2clique[atom].append(c)

    # Merge rings that share more than 2 atoms as they form bridged compounds.
    for c1 in range(len(cliques)):
        for atom in cliques[c1]:
            for c2 in atom2clique[atom]:
                if c1 >= c2 or len(cliques[c1]) <= 2 or len(cliques[c2]) <= 2:
                    continue
                if len(set(cliques[c1]) & set(cliques[c2])) > 2:
                    cliques[c1] = set(cliques[c1]) | set(cliques[c2])
                    xs[c1] = 2
                    cliques[c2] = []
                    xs[c2] = -1
    cliques = [c for c in cliques if len(c) > 0]
    xs = [x for x in xs if x >= 0]

    # Update `atom2clique` mappings.
    atom2clique = [[] for i in range(mol.GetNumAtoms())]
    for c in range(len(cliques)):
        for atom in cliques[c]:
            atom2clique[atom].append(c)

    # Add singleton cliques in case there are more than 2 intersecting
    # cliques. We further compute the "initial" clique graph.
    edges = {}
    for atom in range(mol.GetNumAtoms()):
        cs = atom2clique[atom]
        if len(cs) <= 1:
            continue

        # Number of bond clusters that the atom lies in.
        bonds = [c for c in cs if len(cliques[c]) == 2]
        # Number of ring clusters that the atom lies in.
        rings = [c for c in cs if len(cliques[c]) > 4]

        if len(bonds) > 2 or (len(bonds) == 2 and len(cs) > 2):
            cliques.append([atom])
            xs.append(3)
            c2 = len(cliques) - 1
            for c1 in cs:
                edges[(c1, c2)] = 1

        elif len(rings) > 2:
            cliques.append([atom])
            xs.append(3)
            c2 = len(cliques) - 1
            for c1 in cs:
                edges[(c1, c2)] = 99

        else:
            for i in range(len(cs)):
                for j in range(i + 1, len(cs)):
                    c1, c2 = cs[i], cs[j]
                    count = len(set(cliques[c1]) & set(cliques[c2]))
                    edges[(c1, c2)] = min(count, edges.get((c1, c2), 99))

    # Update `atom2clique` mappings.
    atom2clique = [[] for i in range(mol.GetNumAtoms())]
    for c in range(len(cliques)):
        for atom in cliques[c]:
            atom2clique[atom].append(c)

    if len(edges) > 0:
        edge_index_T, weight = zip(*edges.items())
        edge_index = torch.tensor(edge_index_T).t()
        inv_weight = 100 - torch.tensor(weight)
        graph = to_scipy_sparse_matrix(edge_index, inv_weight, len(cliques))
        junc_tree = minimum_spanning_tree(graph)
        edge_index, _ = from_scipy_sparse_matrix(junc_tree)
        edge_index = to_undirected(edge_index, num_nodes=len(cliques))
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    rows = [[i] * len(atom2clique[i]) for i in range(mol.GetNumAtoms())]
    row = torch.tensor(list(chain.from_iterable(rows)))
    col = torch.tensor(list(chain.from_iterable(atom2clique)))
    atom2clique = torch.stack([row, col], dim=0).to(torch.long)

    if return_vocab:
        vocab = torch.tensor(xs, dtype=torch.long)
        return edge_index, atom2clique, len(cliques), vocab
    else:
        return edge_index, atom2clique, len(cliques)
    

###

def smiles2graph(m_str):
    mgd = MoleculeGraphDataset(halogen_detail=False)
    mol = Chem.MolFromSmiles(m_str)
    #mol = get_mol(m_str)
    atom_feature, bond_feature = mgd.featurize(mol,'atom_full_feature')
    atom_idx, _ = mgd.featurize(mol,'atom_type')
    tree = mgd.junction_tree(mol)

    out_dict = {
        'smiles':m_str,
        'atom_feature':torch.tensor(atom_feature),#.to(torch.int8),
        'atom_types':'|'.join([i.GetSymbol() for i in mol.GetAtoms()]),
        'atom_idx':torch.tensor(atom_idx),#.to(torch.int8),
        'bond_feature':torch.tensor(bond_feature),#.to(torch.int8),

    }
    tree['tree_edge_index'] = tree['tree_edge_index']#.to(torch.int8)
    tree['atom2clique_index'] = tree['atom2clique_index']#.to(torch.int8)
    tree['x_clique'] = tree['x_clique']#.to(torch.int8)

    out_dict.update(tree)
    
    return out_dict 
####

###
def clique_node_features(degree, total_num_Hs, is_aromatic):
    encoding = one_of_k_encoding_unk(degree, [0,1,2,3,4,5,6,7,8,9,10]) + one_of_k_encoding_unk(total_num_Hs, [0,1,2,3,4,5,6,7,8,9,10])
    encoding += [0,0,0,0,0,0,0,0,0,0,1]
    encoding += [0,0,0,0,0,1]
    encoding += [is_aromatic]
    encoding += [0, 0, 0]
    return np.array(encoding)

def smiles2graph_junction_tree(m_str):
    mgd = MoleculeGraphDataset(halogen_detail=False)
    mol = Chem.MolFromSmiles(m_str)
    atom_feature, bond_feature = mgd.featurize(mol,'atom_full_feature')
    atom_idx, _ = mgd.featurize(mol,'atom_type')
    tree = mgd.junction_tree(mol)

    num_atoms = atom_feature.shape[0]
    num_cliques = len(tree['x_clique'])
    c_degree = (torch.bincount(tree['tree_edge_index'][1]) if num_cliques>1 else 0) + torch.bincount(tree['atom2clique_index'][1])
    c_ring = tree['x_clique']!=1
    c_Hs = [0] * num_cliques
    for i, a in enumerate(tree['atom2clique_index'][0]):
       c_Hs[tree['atom2clique_index'][1][i]] += np.sum(atom_feature[a][11:22] * np.arange(11))

    hi_node_feature = np.concatenate([
        atom_feature,
        np.array([clique_node_features(c_degree[i], c_Hs[i], c_ring[i]) for i in range(num_cliques)]),
        np.array([clique_node_features(num_cliques, np.sum(c_Hs), 0)])
    ])

    hi_bond_feature = np.pad(bond_feature, pad_width=((0, num_cliques+1),(0, num_cliques+1)), mode='constant', constant_values=0)
    for i in range(num_atoms):
        a_idx = tree['atom2clique_index'][0][i]
        c_idx = tree['atom2clique_index'][1][i]
        hi_bond_feature[a_idx][c_idx+num_atoms] = hi_bond_feature[c_idx+num_atoms][a_idx] = 5 
        
    for i in range(num_cliques):
        hi_bond_feature[i+num_atoms][num_cliques+num_atoms] = hi_bond_feature[num_cliques+num_atoms][i+num_atoms] = 6

    max_idx = max(list(mgd.ATOM_CODES.values()))
    hi_node_idx = np.concatenate([
        atom_idx,
        np.array([max_idx + 1] * num_cliques).reshape(-1, 1),
        np.array([max_idx + 2]).reshape(-1, 1),
    ])

    node_levels = np.array([0]*num_atoms + [1]*num_cliques + [2])

    out_dict = {
        'smiles':m_str,
        'atom_feature':torch.tensor(hi_node_feature),
        'atom_types':'|'.join([i.GetSymbol() for i in mol.GetAtoms()]+[f'C{i}' for i in tree['x_clique']]+['M']),
        'atom_idx':torch.tensor(hi_node_idx),
        'bond_feature':torch.tensor(hi_bond_feature),
        'node_levels': torch.tensor(node_levels)
    }

    return out_dict 
####

def smiles2graph_BRICS(m_str):
    mgd = MoleculeGraphDataset(halogen_detail=False)
    mol = Chem.MolFromSmiles(m_str)
    atom_feature, bond_feature = mgd.featurize(mol,'atom_full_feature')
    atom_idx, _ = mgd.featurize(mol,'atom_type')
    cliques = motif_decomp(mol)

    num_atoms = atom_feature.shape[0]
    num_cliques = len(cliques)
    if num_cliques==0:
        try:
            return smiles2graph_junction_tree(m_str)
        except Exception or RuntimeError:
            print(f'illed SMILES: {m_str}')
            return None
    else:
        c_degree = [len(c)+1 for c in cliques]
        c_Hs = [0] * num_cliques
        for i, c in enumerate(cliques):
            for a in c:
                c_Hs[i] += np.sum(atom_feature[a][11:22] * np.arange(11))

        hi_node_feature = np.concatenate([
            atom_feature,
            np.array([clique_node_features(c_degree[i], c_Hs[i], 0) for i in range(num_cliques)]),
            np.array([clique_node_features(num_cliques, np.sum(c_Hs), 0)])
        ])

        hi_bond_feature = np.pad(bond_feature, pad_width=((0, num_cliques+1),(0, num_cliques+1)), mode='constant', constant_values=0)
        for i, c in enumerate(cliques):
            hi_bond_feature[i+num_atoms][num_cliques+num_atoms] = hi_bond_feature[num_cliques+num_atoms][i+num_atoms] = 5
            for a in c:
                hi_bond_feature[a][i+num_atoms] = hi_bond_feature[i+num_atoms][a] = 5 

        max_idx = max(list(mgd.ATOM_CODES.values()))
        hi_node_idx = np.concatenate([
            atom_idx,
            np.array([max_idx+1]*num_cliques).reshape(-1, 1),
            np.array([max_idx+2]).reshape(-1, 1),
        ])

        node_levels = np.array([0]*num_atoms + [1]*num_cliques + [2])

        out_dict = {
            'smiles':m_str,
            'atom_feature':torch.tensor(hi_node_feature),
            'atom_types':'|'.join([i.GetSymbol() for i in mol.GetAtoms()]+[f'BC{i}' for i in range(num_cliques)]+['M']),
            'atom_idx':torch.tensor(hi_node_idx),
            'bond_feature':torch.tensor(hi_bond_feature),
            'node_levels': torch.tensor(node_levels),
        }

        return out_dict 

def ligand_init(smiles_list, mode=None):
    ligand_dict = {}
    for smiles in tqdm(smiles_list):
        if mode=='BRICS':
            ligand_dict[smiles] = smiles2graph_BRICS(smiles)
        elif mode=='tree':
            ligand_dict[smiles] = smiles2graph_junction_tree(smiles)
        else:
            ligand_dict[smiles] = smiles2graph(smiles)

    return ligand_dict

import numpy as np
import pandas as pd
import sys

from tqdm import tqdm

import torch
import esm
from torch_geometric.utils import degree, add_self_loops, subgraph, to_undirected, remove_self_loops, coalesce

import math

def protein_init(seqs):
    result_dict = {}
    model_location = "esm2_t33_650M_UR50D"
    model, alphabet = esm.pretrained.load_model_and_alphabet(model_location)
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    batch_converter = alphabet.get_batch_converter()

    for seq in tqdm(seqs):
        seq_feat = seq_feature(seq)
        token_repr, contact_map_proba, logits = esm_extract(
            model, batch_converter, seq, layer=33, approach='last', dim=1280
        )
        assert len(contact_map_proba) == len(seq)
        edge_index, edge_weight = contact_map(contact_map_proba)
        result_dict[seq] = {
            'seq': seq,
            'seq_feat': torch.from_numpy(seq_feat),
            'token_representation': token_repr.half(),
            'num_nodes': len(seq),
            'num_pos': torch.arange(len(seq)).reshape(-1, 1),
            'edge_index': edge_index,
            'edge_weight': edge_weight,
        }

    del model, batch_converter
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result_dict



# normalize
def dic_normalize(dic):
    # print(dic)
    max_value = dic[max(dic, key=dic.get)]
    min_value = dic[min(dic, key=dic.get)]
    # print(max_value)
    interval = float(max_value) - float(min_value)
    for key in dic.keys():
        dic[key] = (dic[key] - min_value) / interval
    dic['X'] = (max_value + min_value) / 2.0
    return dic


pro_res_table = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y',
                 'X']

pro_res_aliphatic_table = ['A', 'I', 'L', 'M', 'V']
pro_res_aromatic_table = ['F', 'W', 'Y']
pro_res_polar_neutral_table = ['C', 'N', 'Q', 'S', 'T']
pro_res_acidic_charged_table = ['D', 'E']
pro_res_basic_charged_table = ['H', 'K', 'R']

res_weight_table = {'A': 71.08, 'C': 103.15, 'D': 115.09, 'E': 129.12, 'F': 147.18, 'G': 57.05, 'H': 137.14,
                    'I': 113.16, 'K': 128.18, 'L': 113.16, 'M': 131.20, 'N': 114.11, 'P': 97.12, 'Q': 128.13,
                    'R': 156.19, 'S': 87.08, 'T': 101.11, 'V': 99.13, 'W': 186.22, 'Y': 163.18}

res_pka_table = {'A': 2.34, 'C': 1.96, 'D': 1.88, 'E': 2.19, 'F': 1.83, 'G': 2.34, 'H': 1.82, 'I': 2.36,
                 'K': 2.18, 'L': 2.36, 'M': 2.28, 'N': 2.02, 'P': 1.99, 'Q': 2.17, 'R': 2.17, 'S': 2.21,
                 'T': 2.09, 'V': 2.32, 'W': 2.83, 'Y': 2.32}

res_pkb_table = {'A': 9.69, 'C': 10.28, 'D': 9.60, 'E': 9.67, 'F': 9.13, 'G': 9.60, 'H': 9.17,
                 'I': 9.60, 'K': 8.95, 'L': 9.60, 'M': 9.21, 'N': 8.80, 'P': 10.60, 'Q': 9.13,
                 'R': 9.04, 'S': 9.15, 'T': 9.10, 'V': 9.62, 'W': 9.39, 'Y': 9.62}

res_pkx_table = {'A': 0.00, 'C': 8.18, 'D': 3.65, 'E': 4.25, 'F': 0.00, 'G': 0, 'H': 6.00,
                 'I': 0.00, 'K': 10.53, 'L': 0.00, 'M': 0.00, 'N': 0.00, 'P': 0.00, 'Q': 0.00,
                 'R': 12.48, 'S': 0.00, 'T': 0.00, 'V': 0.00, 'W': 0.00, 'Y': 0.00}

res_pl_table = {'A': 6.00, 'C': 5.07, 'D': 2.77, 'E': 3.22, 'F': 5.48, 'G': 5.97, 'H': 7.59,
                'I': 6.02, 'K': 9.74, 'L': 5.98, 'M': 5.74, 'N': 5.41, 'P': 6.30, 'Q': 5.65,
                'R': 10.76, 'S': 5.68, 'T': 5.60, 'V': 5.96, 'W': 5.89, 'Y': 5.96}

res_hydrophobic_ph2_table = {'A': 47, 'C': 52, 'D': -18, 'E': 8, 'F': 92, 'G': 0, 'H': -42, 'I': 100,
                             'K': -37, 'L': 100, 'M': 74, 'N': -41, 'P': -46, 'Q': -18, 'R': -26, 'S': -7,
                             'T': 13, 'V': 79, 'W': 84, 'Y': 49}

res_hydrophobic_ph7_table = {'A': 41, 'C': 49, 'D': -55, 'E': -31, 'F': 100, 'G': 0, 'H': 8, 'I': 99,
                             'K': -23, 'L': 97, 'M': 74, 'N': -28, 'P': -46, 'Q': -10, 'R': -14, 'S': -5,
                             'T': 13, 'V': 76, 'W': 97, 'Y': 63}

res_weight_table = dic_normalize(res_weight_table)
res_pka_table = dic_normalize(res_pka_table)
res_pkb_table = dic_normalize(res_pkb_table)
res_pkx_table = dic_normalize(res_pkx_table)
res_pl_table = dic_normalize(res_pl_table)
res_hydrophobic_ph2_table = dic_normalize(res_hydrophobic_ph2_table)
res_hydrophobic_ph7_table = dic_normalize(res_hydrophobic_ph7_table)


def residue_features(residue):
    res_property1 = [1 if residue in pro_res_aliphatic_table else 0, 1 if residue in pro_res_aromatic_table else 0,
                     1 if residue in pro_res_polar_neutral_table else 0,
                     1 if residue in pro_res_acidic_charged_table else 0,
                     1 if residue in pro_res_basic_charged_table else 0]
    res_property2 = [res_weight_table[residue], res_pka_table[residue], res_pkb_table[residue], res_pkx_table[residue],
                     res_pl_table[residue], res_hydrophobic_ph2_table[residue], res_hydrophobic_ph7_table[residue]]
    # print(np.array(res_property1 + res_property2).shape)
    return np.array(res_property1 + res_property2)


# one ont encoding
def one_of_k_encoding(x, allowable_set):
    if x not in allowable_set:
        # print(x)
        raise Exception('input {0} not in allowable set{1}:'.format(x, allowable_set))
    return list(map(lambda s: x == s, allowable_set))


def one_of_k_encoding_unk(x, allowable_set):
    '''Maps inputs not in the allowable set to the last element.'''
    if x not in allowable_set:
        x = allowable_set[-1]
    return list(map(lambda s: x == s, allowable_set))



def seq_feature(pro_seq):    
    if 'U' in pro_seq or 'B' in pro_seq:
        print('U or B in Sequence')
    pro_seq = pro_seq.replace('U','X').replace('B','X')
    pro_hot = np.zeros((len(pro_seq), len(pro_res_table)))
    pro_property = np.zeros((len(pro_seq), 12))
    for i in range(len(pro_seq)):
        # if 'X' in pro_seq:
        #     print(pro_seq)
        pro_hot[i,] = one_of_k_encoding(pro_seq[i], pro_res_table)
        pro_property[i,] = residue_features(pro_seq[i])
    return np.concatenate((pro_hot, pro_property), axis=1)

def contact_map(contact_map_proba, contact_threshold=0.5):
    """Build contact edges without torch_geometric coalesce (no torch-scatter required)."""
    if torch.is_tensor(contact_map_proba):
        cm = contact_map_proba.detach().cpu().float().numpy()
    else:
        cm = np.asarray(contact_map_proba, dtype=np.float32)

    n = int(cm.shape[0])
    if n == 0:
        return torch.zeros((2, 0), dtype=torch.long), torch.zeros(0, dtype=torch.float)

    edges = {}

    def add_edge(i, j, w):
        key = (int(i), int(j))
        edges[key] = max(edges.get(key, 0.0), float(w))

    rows, cols = np.where(cm >= contact_threshold)
    for i, j in zip(rows, cols):
        add_edge(i, j, cm[i, j])

    for i in range(n - 1):
        add_edge(i, i + 1, contact_threshold)
        add_edge(i + 1, i, contact_threshold)
    for i in range(n - 2):
        add_edge(i, i + 2, contact_threshold)
        add_edge(i + 2, i, contact_threshold)

    sym = {}
    for (i, j), w in list(edges.items()):
        if i == j:
            continue
        w_max = max(w, edges.get((j, i), w))
        sym[(i, j)] = w_max
        sym[(j, i)] = w_max

    for i in range(n):
        sym[(i, i)] = max(sym.get((i, i), 0.0), 1.0)

    pairs = list(sym.keys())
    edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()
    edge_weight = torch.tensor([sym[p] for p in pairs], dtype=torch.float)
    return edge_index, edge_weight



def esm_extract(model, batch_converter, seq, layer=36, approach='mean',dim=2560):
    pro_id = 'A'
    if len(seq) <= 700:
        data = []
        data.append((pro_id, seq))
        batch_labels, batch_strs, batch_tokens = batch_converter(data)
        batch_tokens = batch_tokens.to(next(model.parameters()).device, non_blocking=True)

        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[i for i in range(1, layer + 1)], return_contacts=True)

        logits = results["logits"][0].cpu().numpy()[1: len(seq) + 1]
        contact_prob_map = results["contacts"][0].cpu().numpy()
        token_representation = torch.cat([results['representations'][i] for i in range(1, layer + 1)])
        assert token_representation.size(0) == layer

        if approach == 'last':
            token_representation = token_representation[-1]
        elif approach == 'sum':
            token_representation = token_representation.sum(dim=0)
        elif approach == 'mean':
            token_representation = token_representation.mean(dim=0)

        token_representation = token_representation.cpu().numpy()
        token_representation = token_representation[1: len(seq) + 1]
    else:
        contact_prob_map = np.zeros((len(seq), len(seq)))  # global contact map prediction
        token_representation = np.zeros((len(seq), dim))
        logits = np.zeros((len(seq),layer))
        interval = 350
        i = math.ceil(len(seq) / interval)
        # ======================
        # =                    =
        # =                    =
        # =          ======================
        # =          =*********=          =
        # =          =*********=          =
        # ======================          =
        #            =                    =
        #            =                    =
        #            ======================
        # where * is the overlapping area
        # subsection seq contact map prediction
        for s in range(i):
            start = s * interval  # sub seq predict start
            end = min((s + 2) * interval, len(seq))  # sub seq predict end
            sub_seq_len = end - start

            # prediction
            temp_seq = seq[start:end]
            temp_data = []
            temp_data.append((pro_id, temp_seq))
            batch_labels, batch_strs, batch_tokens = batch_converter(temp_data)
            batch_tokens = batch_tokens.to(next(model.parameters()).device, non_blocking=True)
            with torch.no_grad():
                results = model(batch_tokens, repr_layers=[i for i in range(1, layer + 1)], return_contacts=True)

            # insert into the global contact map
            row, col = np.where(contact_prob_map[start:end, start:end] != 0)
            row = row + start
            col = col + start
            contact_prob_map[start:end, start:end] = contact_prob_map[start:end, start:end] + results["contacts"][
                0].cpu().numpy()
            contact_prob_map[row, col] = contact_prob_map[row, col] / 2.0
            
            logits[start:end] += results['logits'][0].cpu().numpy()[1: len(temp_seq) + 1]
            logits[row] = logits[row]/2.0

            ## TOKEN
            subtoken_repr = torch.cat([results['representations'][i] for i in range(1, layer + 1)])
            assert subtoken_repr.size(0) == layer
            if approach == 'last':
                subtoken_repr = subtoken_repr[-1]
            elif approach == 'sum':
                subtoken_repr = subtoken_repr.sum(dim=0)
            elif approach == 'mean':
                subtoken_repr = subtoken_repr.mean(dim=0)

            subtoken_repr = subtoken_repr.cpu().numpy()
            subtoken_repr = subtoken_repr[1: len(temp_seq) + 1]

            trow = np.where(token_representation[start:end].sum(axis=-1) != 0)[0]
            trow = trow + start
            token_representation[start:end] = token_representation[start:end] + subtoken_repr
            token_representation[trow] = token_representation[trow] / 2.0

            if end == len(seq):
                break

    return torch.from_numpy(token_representation), torch.from_numpy(contact_prob_map), torch.from_numpy(logits)




def generate_ESM_structure(model, filename, sequence):
    model.set_chunk_size(256)
    chunk_size = 256
    output = None

    while output is None:
        try:
            with torch.no_grad():
                output = model.infer_pdb(sequence)

            with open(filename, "w") as f:
                f.write(output)
                print("saved", filename)
        except RuntimeError as e:
            if 'out of memory' in str(e):
                print('| WARNING: ran out of memory on chunk_size', chunk_size)
                for p in model.parameters():
                    if p.grad is not None:
                        del p.grad  # free some memory
                torch.cuda.empty_cache()
                chunk_size = chunk_size // 2
                if chunk_size > 2:
                    model.set_chunk_size(chunk_size)
                else:
                    print("Not enough memory for ESMFold")
                    break
            else:
                raise e
    return output is not None




import joblib
from sklearn.preprocessing import StandardScaler

def build_extra_feature_dicts(df, drug_id_col='Drug_ID', target_id_col='Target_ID'):
    drug_cols, prot_cols = get_extra_feature_columns(df)
    drug_extra, prot_extra = {}, {}

    if drug_cols:
        drug_df = df[[drug_id_col] + drug_cols].drop_duplicates(drug_id_col)
        scaler = StandardScaler()
        values = scaler.fit_transform(drug_df[drug_cols].fillna(0.0).astype(float))
        for i, row in drug_df.reset_index(drop=True).iterrows():
            drug_extra[row[drug_id_col]] = torch.tensor(values[i], dtype=torch.float).view(1, -1)

    if prot_cols:
        prot_df = df[[target_id_col] + prot_cols].drop_duplicates(target_id_col)
        scaler = StandardScaler()
        values = scaler.fit_transform(prot_df[prot_cols].fillna(0.0).astype(float))
        for i, row in prot_df.reset_index(drop=True).iterrows():
            prot_extra[row[target_id_col]] = torch.tensor(values[i], dtype=torch.float).view(1, -1)

    print(f"Extra drug features ({len(drug_cols)}): {drug_cols}")
    print(f"Extra protein features ({len(prot_cols)}): {prot_cols}")
    return drug_extra, prot_extra, len(drug_cols), len(prot_cols)

def load_or_build_graph_caches(df, dataset_name, data_dir, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    ligand_cache = os.path.join(save_dir, f'{dataset_name}-ligand-hi.pkl')
    protein_cache = os.path.join(save_dir, f'{dataset_name}-protein.pkl')

    for candidate in [
        ligand_cache,
        os.path.join(data_dir, f'{dataset_name}-ligand-hi.pkl'),
        os.path.join(data_dir, 'molecules.pkl'),
    ]:
        if os.path.exists(candidate):
            print(f'Loading ligand graphs from {candidate}')
            with open(candidate, 'rb') as f:
                mol_dict = pickle.load(f)
            break
    else:
        print('Building BRICS ligand graphs with RDKit...')
        smiles_map = df[['Drug_ID', 'drug_smiles']].drop_duplicates('Drug_ID')
        ligand_dict = ligand_init(smiles_map['drug_smiles'].tolist(), mode='BRICS')
        mol_dict = {}
        for drug_id, smiles in zip(smiles_map['Drug_ID'], smiles_map['drug_smiles']):
            graph = ligand_dict.get(smiles)
            if graph is not None:
                mol_dict[drug_id] = graph
        joblib.dump(mol_dict, ligand_cache)
        print(f'Saved ligand cache to {ligand_cache}')

    for candidate in [
        protein_cache,
        os.path.join(data_dir, f'{dataset_name}-protein.pkl'),
        os.path.join(data_dir, 'proteins.pt'),
    ]:
        if os.path.exists(candidate):
            print(f'Loading protein graphs from {candidate}')
            if candidate.endswith('.pt'):
                prot_dict = torch.load(candidate, map_location='cpu')
            else:
                prot_dict = joblib.load(candidate)
            break
    else:
        print('Building protein graphs with ESM-2 (first run may take a while)...')
        seq_map = df[['Target_ID', 'target_sequence']].drop_duplicates('Target_ID')
        seq_to_target = {row['target_sequence']: row['Target_ID'] for _, row in seq_map.iterrows()}
        protein_by_seq = protein_init(seq_map['target_sequence'].tolist())
        prot_dict = {seq_to_target[seq]: protein_by_seq[seq] for seq in protein_by_seq}
        joblib.dump(prot_dict, protein_cache)
        print(f'Saved protein cache to {protein_cache}')

    return mol_dict, prot_dict

print('Building molecule/protein dictionaries and tabular extras...')
drug_extra_dict, prot_extra_dict, DRUG_EXTRA_DIM, PROT_EXTRA_DIM = build_extra_feature_dicts(df)
mol_dict, prot_dict = load_or_build_graph_caches(df, DATASET.lower(), DATA_DIR, SAVE_DIR)

train_df = train_df[train_df['Drug_ID'].isin(mol_dict.keys()) & train_df['Target_ID'].isin(prot_dict.keys())]
val_df = val_df[val_df['Drug_ID'].isin(mol_dict.keys()) & val_df['Target_ID'].isin(prot_dict.keys())]
test_df = test_df[test_df['Drug_ID'].isin(mol_dict.keys()) & test_df['Target_ID'].isin(prot_dict.keys())]

print(f'\nFinal dataset sizes: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}')

# Free GPU memory after ESM/RDKit preprocessing (critical on Kaggle T4)
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU memory after cache build: {_free / 1e9:.2f} GB free / {_total / 1e9:.2f} GB total')


Building molecule/protein dictionaries and tabular extras...
Extra drug features (1): ['molecular_weight']
Extra protein features (9): ['isoform_count', 'ft_transmem_count', 'pdb_count', 'aac_A', 'aac_R', 'paac_APAAC1', 'paac_APAAC2', 'ctd__PolarizabilityD1075', 'ctd__PolarizabilityD1100']
Loading ligand graphs from d:\Thesis\DTI_Models\Runners\MIF-DTI\dti_run\kiba-ligand-hi.pkl
Loading protein graphs from d:\Thesis\DTI_Models\Runners\MIF-DTI\dti_run\kiba-protein.pkl

Final dataset sizes: Train=67759, Val=22607, Test=22601
GPU memory after cache build: 7.16 GB free / 8.59 GB total


In [90]:
# Create pair lists in the format expected by the dataset
def create_pair_list(dataframe):
    """Create list of pairs in format: 'drug_id target_id label'"""
    pairs = []
    for _, row in dataframe.iterrows():
        pair_str = f"{row['Drug_ID']} {row['Target_ID']} {row['binary_label']}"
        pairs.append(pair_str)
    return pairs

train_pairs = create_pair_list(train_df)
val_pairs = create_pair_list(val_df)
test_pairs = create_pair_list(test_df)

print(f"Created {len(train_pairs)} training pairs")
print(f"Created {len(val_pairs)} validation pairs")
print(f"Created {len(test_pairs)} test pairs")
print(f"\nExample pair: {train_pairs[0]}")

Created 67759 training pairs
Created 22607 validation pairs
Created 22601 test pairs

Example pair: CHEMBL1983025 P06241 0


In [91]:
# Create PyTorch Geometric datasets
print("Creating PyTorch Geometric datasets...")

train_dataset = ProteinMoleculeDataset(
    train_pairs, mol_dict, prot_dict, device=str(DEVICE),
    drug_extra=drug_extra_dict, prot_extra=prot_extra_dict,
    drug_extra_dim=DRUG_EXTRA_DIM, prot_extra_dim=PROT_EXTRA_DIM,
)
valid_dataset = ProteinMoleculeDataset(
    val_pairs, mol_dict, prot_dict, device=str(DEVICE),
    drug_extra=drug_extra_dict, prot_extra=prot_extra_dict,
    drug_extra_dim=DRUG_EXTRA_DIM, prot_extra_dim=PROT_EXTRA_DIM,
)
test_dataset = ProteinMoleculeDataset(
    test_pairs, mol_dict, prot_dict, device=str(DEVICE),
    drug_extra=drug_extra_dict, prot_extra=prot_extra_dict,
    drug_extra_dim=DRUG_EXTRA_DIM, prot_extra_dim=PROT_EXTRA_DIM,
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Valid dataset size: {len(valid_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")


Creating PyTorch Geometric datasets...
Train dataset size: 67759
Valid dataset size: 22607
Test dataset size: 22601


In [92]:
# (deprecated) preprocessed pickle paths are auto-detected in the graph cache cell above.


In [93]:
# Data loaders — adjust batch for GPU VRAM (m2p bipartite graphs are memory-heavy)
if os.environ.get("MIF_DTI_ENV") == "local_conda":
    BATCH_SIZE = globals().get("SUGGESTED_BATCH_SIZE", 8)
    GRAD_ACCUM_STEPS = globals().get("SUGGESTED_GRAD_ACCUM", 16)
else:
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 32
EVAL_BATCH_SIZE = BATCH_SIZE

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    follow_batch=['mol_x', 'clique_x', 'prot_node_aa'], num_workers=0,
)
valid_loader = DataLoader(
    valid_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, drop_last=False,
    follow_batch=['mol_x', 'clique_x', 'prot_node_aa'], num_workers=0,
)
test_loader = DataLoader(
    test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, drop_last=False,
    follow_batch=['mol_x', 'clique_x', 'prot_node_aa'], num_workers=0,
)

print(f"Train micro-batch: {BATCH_SIZE}, grad accum: {GRAD_ACCUM_STEPS} (effective {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Test batches: {len(test_loader)}")


Train micro-batch: 4, grad accum: 32 (effective 128)
Train batches: 16939
Valid batches: 5652
Test batches: 5651


In [94]:
# Test data loading by checking one batch

sample_batch = next(iter(train_loader))
print("Sample batch attributes:")
print(f"mol_x shape: {sample_batch.mol_x.shape}")
print(f"mol_x_feat shape: {sample_batch.mol_x_feat.shape}")
print(f"mol_node_levels present: {hasattr(sample_batch, 'mol_node_levels') and sample_batch.mol_node_levels is not None}")
print(f"prot_node_aa shape: {sample_batch.prot_node_aa.shape}")
print(f"prot_node_evo shape: {sample_batch.prot_node_evo.shape}")
print(f"prot_edge_index shape: {sample_batch.prot_edge_index.shape}")
print(f"cls_y shape: {sample_batch.cls_y.shape}")
print(f"m2p_edge_index shape: {sample_batch.m2p_edge_index.shape}")
if hasattr(sample_batch, 'drug_extra_feat'):
    print(f"drug_extra_feat shape: {sample_batch.drug_extra_feat.shape}")
if hasattr(sample_batch, 'prot_extra_feat'):
    print(f"prot_extra_feat shape: {sample_batch.prot_extra_feat.shape}")

Sample batch attributes:
mol_x shape: torch.Size([154, 1])
mol_x_feat shape: torch.Size([154, 43])
mol_node_levels present: True
prot_node_aa shape: torch.Size([2643, 33])
prot_node_evo shape: torch.Size([2643, 1280])
prot_edge_index shape: torch.Size([2, 17563])
cls_y shape: torch.Size([4])
m2p_edge_index shape: torch.Size([2, 20945])
drug_extra_feat shape: torch.Size([4, 1])
prot_extra_feat shape: torch.Size([4, 9])


In [95]:
# Initialize MIF-DTI model
model = MIFDTI(
    depth=3,
    device=str(DEVICE),
    drug_extra_dim=DRUG_EXTRA_DIM,
    prot_extra_dim=PROT_EXTRA_DIM,
).to(DEVICE)

# Xavier init (official)
weight_p, bias_p = [], []
for p in model.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)
for name, p in model.named_parameters():
    if 'bias' in name:
        bias_p.append(p)
    else:
        weight_p.append(p)

print(f"Model initialized on {DEVICE}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()

with torch.no_grad():
    sample_batch = next(iter(train_loader)).to(DEVICE)
    test_out = model(sample_batch)
    print(f"Forward pass OK: output shape {test_out.shape}, batch size {sample_batch.cls_y.shape[0]}")
    if hasattr(sample_batch, 'm2p_edge_index'):
        print(f"m2p edges in probe batch: {sample_batch.m2p_edge_index.shape[1]:,}")

del sample_batch, test_out
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Model initialized on cuda
Total parameters: 3,436,080
Forward pass OK: output shape torch.Size([4, 2]), batch size 4
m2p edges in probe batch: 17,470


In [96]:
# Training setup (aligned with RunModel.py)
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 50
NUM_EPOCHS = 200

if DATASET.upper() == 'DAVIS':
    weight_ce = torch.tensor([0.3, 0.7], dtype=torch.float, device=DEVICE)
elif DATASET.upper() == 'KIBA':
    weight_ce = torch.tensor([0.2, 0.8], dtype=torch.float, device=DEVICE)
else:
    weight_ce = None

criterion = CELoss(weight_ce, DEVICE)  # re-run this cell after editing CELoss above
optimizer = torch.optim.AdamW(
    [{'params': weight_p, 'weight_decay': WEIGHT_DECAY}, {'params': bias_p, 'weight_decay': 0.0}],
    lr=LEARNING_RATE,
)
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM_STEPS
scheduler = torch.optim.lr_scheduler.CyclicLR(
    optimizer,
    base_lr=LEARNING_RATE,
    max_lr=LEARNING_RATE * 10,
    cycle_momentum=False,
    step_size_up=max(1, len(train_dataset) // EFFECTIVE_BATCH),
)
early_stopper = EarlyStopping(savepath=SAVE_DIR, patience=PATIENCE, verbose=True, delta=0)

USE_AMP = torch.cuda.is_available()
if hasattr(torch.amp, 'GradScaler'):
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
else:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'Training setup complete (AMP={USE_AMP}, effective batch={EFFECTIVE_BATCH})')


Training setup complete (AMP=True, effective batch=128)


In [97]:
# Training / validation helpers
from contextlib import nullcontext
from sklearn.metrics import (
    accuracy_score, average_precision_score,
    precision_score, recall_score, roc_auc_score,
)

def amp_autocast(enabled=True):
    if not enabled:
        return nullcontext()
    if hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast('cuda', enabled=True)
    return torch.cuda.amp.autocast(enabled=True)

def classification_loss(scores, labels):
    """FP32 cross-entropy; safe with AMP (does not use a stale criterion instance)."""
    w = globals().get('weight_ce')
    return F.cross_entropy(scores.float(), labels.long(), weight=w)

def compute_classification_metrics(Y, P, S):
    """ROC-AUC and PR-AUC (average precision); safe for imbalanced / flat scores."""
    Y = np.asarray(Y)
    P = np.asarray(P)
    S = np.asarray(S)
    precision = precision_score(Y, P, zero_division=0)
    recall = recall_score(Y, P, zero_division=0)
    accuracy = accuracy_score(Y, P)
    if len(np.unique(Y)) < 2:
        return accuracy, precision, recall, float('nan'), float('nan')
    return (
        accuracy,
        precision,
        recall,
        roc_auc_score(Y, S),
        average_precision_score(Y, S),
    )

def run_validation(model, loader, criterion, device, max_batches=None):
    model.eval()
    valid_losses, Y, P, S = [], [], [], []
    with torch.no_grad():
        for batch_idx, data in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break
            data = data.to(device)
            with amp_autocast(USE_AMP):
                scores = model(data)
            loss = classification_loss(scores, data.cls_y)
            valid_losses.append(loss.item())

            labels = data.cls_y.cpu().numpy()
            prob = F.softmax(scores.float(), dim=1).cpu().numpy()
            preds = np.argmax(prob, axis=1)
            pos_scores = prob[:, 1]

            Y.extend(labels)
            P.extend(preds)
            S.extend(pos_scores)

    accuracy, precision, recall, roc_auc, prc = compute_classification_metrics(Y, P, S)
    return np.mean(valid_losses), accuracy, precision, recall, roc_auc, prc

# Smoke test: loss + full metrics path (2 val batches, ~1–2 min) before long training
if 'valid_loader' in globals() and 'model' in globals():
    _b = next(iter(valid_loader)).to(DEVICE)
    with torch.no_grad():
        with amp_autocast(USE_AMP):
            _s = model(_b)
        _loss = classification_loss(_s, _b.cls_y)
    print(f'AMP loss smoke test OK: loss={_loss.item():.4f}')
    _vl, _va, _vp, _vr, _vauc, _vprc = run_validation(
        model, valid_loader, criterion, DEVICE, max_batches=2,
    )
    print(
        f'Metrics smoke test OK (2 batches): loss={_vl:.4f} acc={_va:.4f} '
        f'roc_auc={_vauc:.4f} prc={_vprc:.4f}'
    )
else:
    print('Skip smoke test until model and valid_loader exist')

print('Training helpers defined')


AMP loss smoke test OK: loss=0.6923
Metrics smoke test OK (2 batches): loss=0.6648 acc=0.6250 roc_auc=0.3333 prc=0.6968
Training helpers defined


In [ ]:
# Train MIF-DTI
import gc

print('Starting training...')
if torch.cuda.is_available():
    torch.cuda.empty_cache()

for epoch in range(1, NUM_EPOCHS + 1):
    if early_stopper.early_stop:
        break

    model.train()
    train_losses = []
    optimizer.zero_grad(set_to_none=True)
    accum_step = 0

    for data in train_loader:
        data = data.to(DEVICE)
        with amp_autocast(USE_AMP):
            scores = model(data)
        loss = classification_loss(scores, data.cls_y) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        accum_step += 1
        train_losses.append(loss.item() * GRAD_ACCUM_STEPS)

        if accum_step >= GRAD_ACCUM_STEPS:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            accum_step = 0

        del data, scores, loss

    if accum_step > 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    train_loss = np.mean(train_losses)
    valid_loss, valid_acc, valid_prec, valid_rec, valid_auc, valid_prc = run_validation(
        model, valid_loader, criterion, DEVICE
    )

    print(
        f'[{epoch:>{len(str(NUM_EPOCHS))}}/{NUM_EPOCHS}] '
        f'train_loss: {train_loss:.5f} valid_loss: {valid_loss:.5f} '
        f'valid_AUC: {valid_auc:.5f} valid_PRC: {valid_prc:.5f} '
        f'valid_Accuracy: {valid_acc:.5f} valid_Precision: {valid_prec:.5f} '
        f'valid_Recall: {valid_rec:.5f}'
    )

    early_stopper(valid_acc, model, epoch)

print('Training finished')


Starting training...


c:\Users\Abdullah\anaconda3\envs\dti_research\lib\site-packages\torch_geometric\warnings.py:11: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(message)


[  1/200] train_loss: 0.26536 valid_loss: 0.23391 valid_AUC: 0.78640 valid_PRC: 0.92022 valid_Accuracy: 0.79971 valid_Precision: 0.79903 valid_Recall: 0.99646
Have a new best checkpoint: (-inf --> 0.799708).  Saving model ...


c:\Users\Abdullah\anaconda3\envs\dti_research\lib\site-packages\torch_geometric\warnings.py:11: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(message)


In [ ]:
# Load best checkpoint and evaluate
ckpt_path = f"{SAVE_DIR}/valid_best_checkpoint-{DEVICE}.pth"
if not os.path.exists(ckpt_path):
    ckpt_path = f"{SAVE_DIR}/valid_best_checkpoint.pth"

print(f"Loading best model from {ckpt_path}...")
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))

print('\nEvaluating on validation set:')
valid_results = test_model(
    MODEL=model,
    dataset_loader=valid_loader,
    save_path=SAVE_DIR,
    DATASET=DATASET,
    LOSS=criterion,
    DEVICE=DEVICE,
    dataset_class='Valid',
    save=True,
    FOLD_NUM=1,
    MIF=True,
)

print('\nEvaluating on test set:')
test_results = test_model(
    MODEL=model,
    dataset_loader=test_loader,
    save_path=SAVE_DIR,
    DATASET=DATASET,
    LOSS=criterion,
    DEVICE=DEVICE,
    dataset_class='Test',
    save=True,
    FOLD_NUM=1,
    MIF=True,
)


In [ ]:
# Display final results

print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
print("\nValidation Set:")
print(valid_results[0])
print("\nTest Set:")
print(test_results[0])
print("\n" + "="*70)

# Save summary
with open(f"{SAVE_DIR}/final_results.txt", 'w') as f:
    f.write("FINAL RESULTS\n")
    f.write("="*70 + "\n\n")
    f.write("Validation Set:\n")
    f.write(valid_results[0] + "\n\n")
    f.write("Test Set:\n")
    f.write(test_results[0] + "\n")

print(f"\nResults saved to {SAVE_DIR}/final_results.txt")
print(f"Model checkpoint saved to {SAVE_DIR}/valid_best_checkpoint.pth")


FINAL RESULTS

Validation Set:
Valid: Loss:0.47953;Accuracy:0.79747;Precision:0.80306;Recall:0.98543;AUC:0.67346;PRC:0.87765.

Test Set:
Test: Loss:0.47912;Accuracy:0.79870;Precision:0.80403;Recall:0.98554;AUC:0.67389;PRC:0.87863.


Results saved to /kaggle/working/dti_run/final_results.txt
Model checkpoint saved to /kaggle/working/dti_run/valid_best_checkpoint.pth


In [ ]:
# Optional: List all output files

print("\nOutput files created:")
for file in os.listdir(SAVE_DIR):
    filepath = os.path.join(SAVE_DIR, file)
    if os.path.isfile(filepath):
        size = os.path.getsize(filepath)
        print(f"  {file} ({size} bytes)")


Output files created:
  valid_best_checkpoint.pth (286971 bytes)
  KIBA_Valid_prediction.txt (94128 bytes)
  KIBA_Test_prediction.txt (94128 bytes)
  final_results.txt (302 bytes)
